# DIMER Notebook: Semantic Search and Reranking with Qwen3

**Notebook profile:** `MULTI-CAPABILITY`  
**Pedagogical mode:** `WORKSHOP`  
**DIMER Notebook Specification:** `2.1`  
**Standalone:** yes  
**Default workflow:** frozen inference only — no fine-tuning or adapter creation

This notebook composes two live DIMER models into a measurable two-stage search system:

1. **Qwen3-Embedding-0.6B** retrieves a broad candidate set with reusable dense vectors.
2. **Qwen3-Reranker-0.6B** spends more compute on a small shortlist and reorders it with a cross-encoder relevance score.

### Learning objectives

By the end you should be able to:

- distinguish bi-encoder retrieval from cross-encoder reranking;
- precompute and reuse document embeddings;
- retrieve candidates with cosine similarity;
- rerank only the top-*k* candidates;
- measure the retriever and the composed system separately;
- explain why a reranker cannot recover a document omitted by first-stage retrieval;
- inspect cases where reranking helps, hurts, preserves, or cannot recover the gold result; and
- export machine-readable rankings, metrics, embeddings, and provenance.

> **Evidence boundary.** The default data is a small Banking77-derived tutorial retrieval task. Results from this notebook are sample/tutorial evidence, not a production benchmark.

**AI Use Disclosure:** Generative AI assisted with this notebook’s code and instructional content under maintainer direction. The maintainer remains responsible for review, validation, and release decisions. AI-generated material may contain errors; validation claims are limited to documented runs and configurations. AI use does not imply independent verification, provider endorsement, or release approval.

## How to use this notebook

**Who it is for.** Learners who can run Python cells in Colab/Jupyter and are new to embeddings, semantic search, or reranking.

**Runtime.** A **Linux x86_64** runtime (Google Colab, Kaggle or Linux Jupyter). A CUDA GPU such as a Tesla T4 is recommended; CPU execution is possible but reranking is substantially slower.

**How to run it.**
1. Select the documented runtime/accelerator.
2. Choose **Run all** for the canonical path; the defaults are the reference settings.
3. Read the explanation around each learning stage while the cells execute.
4. Cells marked **Infrastructure** handle setup, model acquisition, provenance, or orchestration. You may run those cells without understanding their implementation.

### Task at a glance

`query → embedding retriever → top-k candidates → cross-encoder reranker → ranked results`

### Roadmap

1. Understand the task, inputs, outputs, and evaluation boundary.
2. Inspect and validate the built-in sample.
3. Establish the simple reference/baseline where applicable.
4. Run the model or model comparison.
5. Inspect errors, disagreements, robustness, or resource tradeoffs.
6. Try one controlled change and explain what changed.
7. Write an evidence-based conclusion, then optionally try your own compatible data.

### What successful execution looks like

You should finish with validated sample/input evidence, the notebook's principal baseline/reference result, model outputs and comparison metrics, at least one diagnostic or qualitative view, and machine-readable results/provenance where the capability supports them. Exact numeric values may vary slightly with the supported runtime; interpret the pattern and the stated metric semantics rather than treating one number as universal.


### Reading layers and active learning

**Core concepts:** Sections 1, 8 and 12 explain representations and pair scoring. **Evaluation practice:** Sections 7, 9–17 and the checkpoints compare results and locate failures. **Infrastructure:** setup, manifests and model runtime classes are collapsed; run them without studying their implementation. Basic Python and Colab familiarity is sufficient; no prior ML experience is assumed.

For active learning, execute section by section and write predictions before revealing scores. Form controls are in Configuration. The optional depth activity uses Section 17’s sweep without changing the canonical `RERANK_K`, rankings or metrics. Its results are exploratory because the same test queries have already been inspected.

Setup installs nothing into the notebook kernel: it builds a separate, locked environment and runs every later code cell there, so a fresh runtime runs through without a session restart (see Section 3).

## 1. How two-stage retrieval works

A dense **embedding retriever** encodes each document independently and can therefore precompute document vectors:

`document → vector`

At query time:

`query → vector → cosine similarity against the index → top-k candidates`

A **cross-encoder reranker** instead evaluates the query and one candidate jointly:

`query + candidate → relevance score`

That richer interaction is more expensive because it requires a model forward pass for every query–candidate pair. A practical search system therefore uses the retriever to reduce the corpus to a shortlist, then applies the reranker only to that shortlist.

The central constraint in this notebook is:

> **The reranker can reorder only the documents it receives. If the relevant document is absent from the first-stage shortlist, it cannot recover it.**

## 2. Configuration

The default values form the canonical `Run all` path. The optional K-sweep and BYOD branches are disabled by default.

In [1]:
USE_BYOD = False  # @param {type:"boolean"}
DOCUMENTS_PATH = ""  # @param {type:"string"}
QUERIES_PATH = ""  # @param {type:"string"}
# Describe your own retrieval task here. Blank reuses the banking-intent instructions below, which fit
# only data with the same task (a support message matched to the intent it expresses).
BYOD_EMBEDDING_INSTRUCTION = ""  # @param {type:"string"}
BYOD_RERANK_INSTRUCTION = ""  # @param {type:"string"}

SAMPLE_SEED = 42  # @param {type:"integer"}
RERANK_K = 6  # @param {type:"integer"}

EMBEDDING_INSTRUCTION = "Given a customer support message, retrieve the banking intent it expresses"
RERANK_INSTRUCTION = "Given a customer support message, judge whether the document names the banking intent it expresses"

RUN_K_SWEEP = False  # @param {type:"boolean"}
K_SWEEP_VALUES = [3, 6, 10]

OUTPUT_DIR = "outputs/qwen3_semantic_search"

if not 2 <= RERANK_K <= 32:
    raise ValueError("RERANK_K must be in 2..32 for this tutorial.")
if RUN_K_SWEEP and any((not isinstance(k, int) or not 2 <= k <= 32) for k in K_SWEEP_VALUES):
    raise ValueError("Every K_SWEEP_VALUES entry must be an integer in 2..32.")

print({
    "USE_BYOD": USE_BYOD,
    "SAMPLE_SEED": SAMPLE_SEED,
    "RERANK_K": RERANK_K,
    "RUN_K_SWEEP": RUN_K_SWEEP,
    "OUTPUT_DIR": OUTPUT_DIR,
})

{'USE_BYOD': False, 'SAMPLE_SEED': 42, 'RERANK_K': 6, 'RUN_K_SWEEP': False, 'OUTPUT_DIR': 'outputs/qwen3_semantic_search'}


## 3. Runtime and reproducibility

The two live DIMER carriers share the same tested dependency set. This notebook uses those exact principal-library pins, in an isolated environment.

The setup cell installs nothing into the notebook kernel. It downloads a size- and SHA-256-pinned `uv`, creates a separate environment (`dimer_isolated_env/`) with a managed CPython 3.12.12, and installs the hash-locked, wheel-only requirements into it (`--require-hashes --only-binary :all:`). From then on every code cell runs in one persistent worker process in that environment; printed output appears below each cell as usual, and the cell records the environment's versions in `RUNTIME`. Hosted kernels such as Colab import NumPy at startup, but nothing the kernel has loaded is replaced, so on a fresh runtime **Run all** normally continues without a restart and never asks for one.

**Platform.** The locked environment is built for manylinux x86_64, so the notebook needs a **Linux x86_64** runtime (Google Colab, Kaggle or Linux Jupyter). On another platform the setup cell stops with that message.

**Recommended runtime:** CUDA GPU such as a Tesla T4. CPU works, but cross-encoder reranking is substantially slower.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [2]:
# @title Infrastructure: build the locked isolated environment and route the remaining cells to it
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
# Nothing is installed into this kernel, so Run all never needs a session restart. The pinned libraries go into a
# separate uv environment (managed CPython 3.12.12, hash-locked wheels only); every later code cell runs there.
import hashlib
import io
import os
import platform
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from multiprocessing.connection import Connection
from pathlib import Path

from IPython import get_ipython as _kernel_shell

# The direct pins: the same versions this notebook pinned when it installed into the kernel.
PINS = [
    "torch==2.14.0",
    "torchvision==0.29.0",
    "torchaudio==2.11.0",
    "transformers==4.57.6",
    "huggingface-hub==0.36.2",
    "safetensors==0.8.0",
    "numpy==2.1.3",
]
MANAGED_PYTHON = "3.12.12"
UV_URL = (
    'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
)
UV_BYTES = 20081404
UV_SHA256 = "aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60"
LOCK_NAME = "requirements-semantic-search-workshop.lock.txt"
LOCK_SHA256 = "b4325c3e2b8fd9135e2c08608a7542b53c52e61528833127889c5fc5e419c214"
LOCKED_PACKAGES = 47
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile tutorials/requirements-semantic-search-workshop.in --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-semantic-search-workshop.lock.txt
# Direct pins (requirements-semantic-search-workshop.in) are the versions the notebook pinned before it moved to this
# isolated environment on 2026-10-03; no version was changed. Every entry is hash-locked and wheel-only.
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
    # via requests
charset-normalizer==3.5.2 \
    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \
    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \
    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \
    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \
    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \
    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \
    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \
    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \
    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \
    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \
    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \
    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \
    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \
    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \
    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \
    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \
    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \
    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \
    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \
    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \
    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \
    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \
    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \
    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \
    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \
    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \
    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \
    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \
    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \
    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \
    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \
    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \
    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \
    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \
    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \
    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \
    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \
    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \
    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \
    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \
    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \
    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \
    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \
    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \
    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \
    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \
    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \
    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \
    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \
    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \
    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \
    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \
    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \
    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \
    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \
    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \
    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \
    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \
    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \
    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \
    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \
    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \
    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \
    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \
    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \
    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \
    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \
    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \
    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \
    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \
    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \
    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \
    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \
    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \
    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \
    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \
    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \
    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \
    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \
    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \
    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \
    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \
    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \
    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \
    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \
    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \
    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \
    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \
    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \
    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \
    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \
    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \
    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \
    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \
    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \
    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \
    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \
    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \
    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \
    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \
    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \
    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \
    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \
    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \
    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \
    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \
    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \
    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \
    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \
    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \
    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \
    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \
    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \
    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \
    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \
    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \
    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \
    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \
    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \
    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \
    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \
    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \
    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \
    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \
    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \
    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \
    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \
    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \
    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \
    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \
    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \
    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \
    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \
    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \
    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \
    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \
    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \
    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \
    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \
    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \
    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \
    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \
    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \
    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \
    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \
    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \
    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \
    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \
    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \
    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \
    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \
    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \
    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \
    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \
    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \
    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \
    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \
    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \
    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \
    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \
    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \
    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \
    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \
    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \
    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \
    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \
    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \
    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \
    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \
    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \
    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \
    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f
    # via requests
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via torch
filelock==4.0.9 \
    --hash=sha256:635e7d67fa92654eed444e75e9ca18426d34e77ad9c469bf4373f75a932f7b22 \
    --hash=sha256:9287fd61b99a806e5202be29a83034c0808a1b9830e820537c2f5773981f7eeb
    # via
    #   huggingface-hub
    #   torch
    #   transformers
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   huggingface-hub
    #   torch
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
    # via huggingface-hub
huggingface-hub==0.36.2 \
    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \
    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270
    # via
    #   -r tutorials/requirements-semantic-search-workshop.in
    #   tokenizers
    #   transformers
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via requests
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via torch
markupsafe==3.0.4 \
    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \
    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \
    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \
    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \
    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \
    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \
    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \
    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \
    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \
    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \
    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \
    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \
    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \
    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \
    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \
    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \
    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \
    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \
    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \
    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \
    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \
    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \
    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \
    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \
    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \
    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \
    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \
    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \
    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \
    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \
    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \
    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \
    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \
    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \
    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \
    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \
    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \
    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \
    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \
    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \
    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \
    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \
    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \
    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \
    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \
    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \
    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \
    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \
    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \
    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \
    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \
    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \
    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \
    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \
    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \
    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \
    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \
    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \
    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \
    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \
    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \
    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \
    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \
    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \
    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \
    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \
    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \
    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \
    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \
    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \
    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \
    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \
    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \
    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \
    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \
    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \
    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \
    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \
    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \
    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \
    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \
    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \
    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \
    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \
    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \
    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \
    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \
    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \
    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \
    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \
    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \
    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \
    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \
    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \
    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \
    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \
    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \
    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \
    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \
    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \
    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \
    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \
    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \
    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \
    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \
    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \
    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \
    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \
    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \
    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \
    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \
    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \
    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \
    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \
    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \
    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \
    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \
    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \
    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \
    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \
    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \
    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \
    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \
    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \
    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \
    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \
    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \
    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \
    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \
    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \
    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \
    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \
    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \
    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \
    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \
    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \
    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \
    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \
    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \
    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \
    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \
    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \
    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \
    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \
    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \
    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \
    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \
    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \
    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \
    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \
    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3
    # via jinja2
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via torch
numpy==2.1.3 \
    --hash=sha256:016d0f6f5e77b0f0d45d77387ffa4bb89816b57c835580c3ce8e099ef830befe \
    --hash=sha256:02135ade8b8a84011cbb67dc44e07c58f28575cf9ecf8ab304e51c05528c19f0 \
    --hash=sha256:08788d27a5fd867a663f6fc753fd7c3ad7e92747efc73c53bca2f19f8bc06f48 \
    --hash=sha256:0d30c543f02e84e92c4b1f415b7c6b5326cbe45ee7882b6b77db7195fb971e3a \
    --hash=sha256:0fa14563cc46422e99daef53d725d0c326e99e468a9320a240affffe87852564 \
    --hash=sha256:13138eadd4f4da03074851a698ffa7e405f41a0845a6b1ad135b81596e4e9958 \
    --hash=sha256:14e253bd43fc6b37af4921b10f6add6925878a42a0c5fe83daee390bca80bc17 \
    --hash=sha256:15cb89f39fa6d0bdfb600ea24b250e5f1a3df23f901f51c8debaa6a5d122b2f0 \
    --hash=sha256:17ee83a1f4fef3c94d16dc1802b998668b5419362c8a4f4e8a491de1b41cc3ee \
    --hash=sha256:2312b2aa89e1f43ecea6da6ea9a810d06aae08321609d8dc0d0eda6d946a541b \
    --hash=sha256:2564fbdf2b99b3f815f2107c1bbc93e2de8ee655a69c261363a1172a79a257d4 \
    --hash=sha256:3522b0dfe983a575e6a9ab3a4a4dfe156c3e428468ff08ce582b9bb6bd1d71d4 \
    --hash=sha256:4394bc0dbd074b7f9b52024832d16e019decebf86caf909d94f6b3f77a8ee3b6 \
    --hash=sha256:45966d859916ad02b779706bb43b954281db43e185015df6eb3323120188f9e4 \
    --hash=sha256:4d1167c53b93f1f5d8a139a742b3c6f4d429b54e74e6b57d0eff40045187b15d \
    --hash=sha256:4f2015dfe437dfebbfce7c85c7b53d81ba49e71ba7eadbf1df40c915af75979f \
    --hash=sha256:50ca6aba6e163363f132b5c101ba078b8cbd3fa92c7865fd7d4d62d9779ac29f \
    --hash=sha256:50d18c4358a0a8a53f12a8ba9d772ab2d460321e6a93d6064fc22443d189853f \
    --hash=sha256:5641516794ca9e5f8a4d17bb45446998c6554704d888f86df9b200e66bdcce56 \
    --hash=sha256:576a1c1d25e9e02ed7fa5477f30a127fe56debd53b8d2c89d5578f9857d03ca9 \
    --hash=sha256:6a4825252fcc430a182ac4dee5a505053d262c807f8a924603d411f6718b88fd \
    --hash=sha256:72dcc4a35a8515d83e76b58fdf8113a5c969ccd505c8a946759b24e3182d1f23 \
    --hash=sha256:747641635d3d44bcb380d950679462fae44f54b131be347d5ec2bce47d3df9ed \
    --hash=sha256:762479be47a4863e261a840e8e01608d124ee1361e48b96916f38b119cfda04a \
    --hash=sha256:78574ac2d1a4a02421f25da9559850d59457bac82f2b8d7a44fe83a64f770098 \
    --hash=sha256:825656d0743699c529c5943554d223c021ff0494ff1442152ce887ef4f7561a1 \
    --hash=sha256:8637dcd2caa676e475503d1f8fdb327bc495554e10838019651b76d17b98e512 \
    --hash=sha256:96fe52fcdb9345b7cd82ecd34547fca4321f7656d500eca497eb7ea5a926692f \
    --hash=sha256:973faafebaae4c0aaa1a1ca1ce02434554d67e628b8d805e61f874b84e136b09 \
    --hash=sha256:996bb9399059c5b82f76b53ff8bb686069c05acc94656bb259b1d63d04a9506f \
    --hash=sha256:a38c19106902bb19351b83802531fea19dee18e5b37b36454f27f11ff956f7fc \
    --hash=sha256:a6b46587b14b888e95e4a24d7b13ae91fa22386c199ee7b418f449032b2fa3b8 \
    --hash=sha256:a9f7f672a3388133335589cfca93ed468509cb7b93ba3105fce780d04a6576a0 \
    --hash=sha256:aa08e04e08aaf974d4458def539dece0d28146d866a39da5639596f4921fd761 \
    --hash=sha256:b0df3635b9c8ef48bd3be5f862cf71b0a4716fa0e702155c45067c6b711ddcef \
    --hash=sha256:b47fbb433d3260adcd51eb54f92a2ffbc90a4595f8970ee00e064c644ac788f5 \
    --hash=sha256:baed7e8d7481bfe0874b566850cb0b85243e982388b7b23348c6db2ee2b2ae8e \
    --hash=sha256:bc6f24b3d1ecc1eebfbf5d6051faa49af40b03be1aaa781ebdadcbc090b4539b \
    --hash=sha256:c006b607a865b07cd981ccb218a04fc86b600411d83d6fc261357f1c0966755d \
    --hash=sha256:c181ba05ce8299c7aa3125c27b9c2167bca4a4445b7ce73d5febc411ca692e43 \
    --hash=sha256:c7662f0e3673fe4e832fe07b65c50342ea27d989f92c80355658c7f888fcc83c \
    --hash=sha256:c80e4a09b3d95b4e1cac08643f1152fa71a0a821a2d4277334c88d54b2219a41 \
    --hash=sha256:c894b4305373b9c5576d7a12b473702afdf48ce5369c074ba304cc5ad8730dff \
    --hash=sha256:d7aac50327da5d208db2eec22eb11e491e3fe13d22653dce51b0f4109101b408 \
    --hash=sha256:d89dd2b6da69c4fff5e39c28a382199ddedc3a5be5390115608345dec660b9e2 \
    --hash=sha256:d9beb777a78c331580705326d2367488d5bc473b49a9bc3036c154832520aca9 \
    --hash=sha256:dc258a761a16daa791081d026f0ed4399b582712e6fc887a95af09df10c5ca57 \
    --hash=sha256:e14e26956e6f1696070788252dcdff11b4aca4c3e8bd166e0df1bb8f315a67cb \
    --hash=sha256:e6988e90fcf617da2b5c78902fe8e668361b43b4fe26dbf2d7b0f8034d4cafb9 \
    --hash=sha256:e711e02f49e176a01d0349d82cb5f05ba4db7d5e7e0defd026328e5cfb3226d3 \
    --hash=sha256:ea4dedd6e394a9c180b33c2c872b92f7ce0f8e7ad93e9585312b0c5a04777a4a \
    --hash=sha256:ecc76a9ba2911d8d37ac01de72834d8849e55473457558e12995f4cd53e778e0 \
    --hash=sha256:f55ba01150f52b1027829b50d70ef1dafd9821ea82905b63936668403c3b471e \
    --hash=sha256:f653490b33e9c3a4c1c01d41bc2aef08f9475af51146e4a7710c450cf9761598 \
    --hash=sha256:fa2d1337dc61c8dc417fbccf20f6d1e139896a30721b7f1e832b2bb6ef4eb6c4
    # via
    #   -r tutorials/requirements-semantic-search-workshop.in
    #   torchvision
    #   transformers
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via cuda-toolkit
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via
    #   huggingface-hub
    #   transformers
pillow==12.3.0 \
    --hash=sha256:00808c5e14ef63ac5161091d242999076604ff74b883423a11e5d7bbb38bf756 \
    --hash=sha256:04f01d28a6aaff387bf842a13be313df23ba0597a44f1a976c9feb3c6ff4711a \
    --hash=sha256:06ff022112bc9cbf83b60f8e028d94ad87b60621706487e65f673de61610ab59 \
    --hash=sha256:0740a512dc522224c77d9aa5a8d70d8b7d73fb91f2c21125d8d025d3b8990e45 \
    --hash=sha256:0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3 \
    --hash=sha256:0dd2064cbc55aaec028ef5fbb60fa47bb6c3e7918e07ff17935284b227a9d2df \
    --hash=sha256:0feb2e9d6ad6c9e3c06effe9d00f3f1e618a6643273576b016f591e9315a7139 \
    --hash=sha256:10e41f0fbf1eec8cfd234b8fe17a4caac7c9d0db4c204d3c173a8f9f6ef3232b \
    --hash=sha256:1182d52bc2d5e5d7d0949503aa7e36d12f42205dc287e4883f407b1988820d39 \
    --hash=sha256:164b31cd1a0490ab6efae01aa5df49da7061be0af1b30e035b6e9a1bfe34ee6e \
    --hash=sha256:1657923d2d45afb66526e5b933e5b3052e6bdea196c90d3abb2424e18c77dae8 \
    --hash=sha256:186941b6aef820ad110fb01fb06eb925374dc3a21b17e37ec9a53b250c6fe2d1 \
    --hash=sha256:1cca606cd25738df4ed873d5ad46bbdb3d83b5cbca291f6b4ff13a4df6b0bbe8 \
    --hash=sha256:21900ce7ba264168cd50defae43cd75d25c833ad4ad6e73ffc5596d12e25ac89 \
    --hash=sha256:236ff70b9312fb68943c703aa842ca6a758abfa45ac187a5e7c1452e96ef72b5 \
    --hash=sha256:23aceaa007d6172b02c277f0cd359c79492bbb14f7072b4ede9fbcaf20648130 \
    --hash=sha256:23d27a3e0307ec2244cc51e7287b919aa68d097504ebe19df4e76a98a3eea5bd \
    --hash=sha256:24870b09b224f7ae3c39ed07d10e819d06f8720bc551847b1d623832b5b0e28d \
    --hash=sha256:251bf95b67017e27b13d82f5b326234ca62d70f9cf4c2b9032de2358a3b12c7b \
    --hash=sha256:25b9b82bb22e6e2b3cd07b39c68b7b862001226cb3dff7130d1cb914121b39ed \
    --hash=sha256:28ce87c5ab450a9dd970b52e5aca5fe63ed432d18a2eaddd1979a00a1ba24ace \
    --hash=sha256:300557495eb45ebb8aec96c2da9c4be642fbf7cd937278b4013ba894ea8eb0eb \
    --hash=sha256:30f2aa603c41533cc25c05acd0da21636e84a315768feb631c937177db558931 \
    --hash=sha256:331b624368d4f1d069149002f25f44bc61c8919ce8ddb3c45bdad8f6e2d89510 \
    --hash=sha256:37d6d0a00072fd2948eb22bce7e1475f34569d90c87c59f7a2ec59541b77f7a6 \
    --hash=sha256:37dc8f7bbb66efe481bb60defacef820c950c24713fb44962ed6aa2a50966de1 \
    --hash=sha256:3b8182a766685eaa002637e28b4ec8d6b18819a0c71f579bf0dbaa5830297cce \
    --hash=sha256:3edce1d53195db527e0191f84b71d02022de0540bf43a16ed734ed7537b07385 \
    --hash=sha256:446c34dcc4324b084a53b705127dc15717b22c5e140ae0a3c38349d4efec071e \
    --hash=sha256:4998562bf62a445225f22e07c896bb04b35b1b1f2eb6d760584c9c51d7a5f78c \
    --hash=sha256:4b0a7fe987b14c31ebda6083f74f22b561fd3739bc0ac51e019622e3d72668c7 \
    --hash=sha256:4e8c2a84d977f50b9daed6eeaf3baef67d00d5d74d932288f02cb94518ee3ace \
    --hash=sha256:4f883547d4b7f0495ebe7056b0cc2aea76094e7a4abc8e933540f3271df27d9c \
    --hash=sha256:514435a37670e3e5e08f3945b68718b6ed329bb84367777e16f9f4dfe1e61a0f \
    --hash=sha256:53aa02d20d10c3d814d536aa4e5ac9b84ca0ff5a88377963b085ad6822f93e64 \
    --hash=sha256:5594fc43d548a7ed94949d139aa1341b270f1863f11cfd37f5a6c8b778a6b67f \
    --hash=sha256:571b9fcb07b97ef3a492028fb3d2dc0993ca23a06138b0315286566d29ef718a \
    --hash=sha256:57b3d78c95ba9059768b10e28b813002261d3f3dfc55cc48b0c988f625175827 \
    --hash=sha256:5afb51d599ea772b8365ae807ae557f18bccfe46ab261fd1c2a9ed700fc6eb17 \
    --hash=sha256:6b02afb9b97f65fbca5f31db6a2a3ba21aa93030225f150fa3f249717e938fb4 \
    --hash=sha256:6c0016e7b354317c4e9e525b937ac8596c38d2d232b419529b9cd7a1cd46e39a \
    --hash=sha256:71d6097b330eea8fd15097780c8e89cb1a8ce7838669f48c5bacd6f663dd4701 \
    --hash=sha256:756c768d0c9c2955feb7a56c37ea24aea2e369f8d36a88da270b6a9f19e62b5e \
    --hash=sha256:78cb2c6865a35ab8ff8b75fd122f6033b92a62c82801110e48ddd6c936a45d91 \
    --hash=sha256:7a743ff716f746fc19a9557f60dab1600d4613255f8a7aeb3cdde4db7eb15a66 \
    --hash=sha256:85f998ea1848bc6757289e739cfbdda3a04adfd58b02fc018ce54d754a5ce468 \
    --hash=sha256:8728f216dcdb6e6d555cf971cb34076139ad74b31fc2c14da4fafc741c5f6217 \
    --hash=sha256:877c3f311ff35410f690861c4409e7ccbf0cd2f878e50628a28e5a0bb689e658 \
    --hash=sha256:8cd2f7bdda092d99c9fc2fb7391354f306d01443d22785d0cbfafa2e2c8bb418 \
    --hash=sha256:8e95e1385e4998ae9694eeaa4730ba5457ff61185b3a55e2e7bea0880aef452a \
    --hash=sha256:962864dc93511324d51ddbb5b9f8731bf71675b93ca612a07441896f4688fb8c \
    --hash=sha256:9cf95fe4d0f84c82d282745d9bb08ad9f926efa00be4697e767b814ce40d4330 \
    --hash=sha256:9e881fca225083806662a5c43d627d215f258ff43c890f831966c7d7ba9c7402 \
    --hash=sha256:a2b55dd6b2a4c4b7d87ffa56bdb33fdc5fdb9a462173861a7bc097f17d91cb09 \
    --hash=sha256:a45650e8ce7fafffd731db8550230db6b0d306d181a90b67d3e6bca2f1990930 \
    --hash=sha256:a876864214e136f0eb367788dbd7df045f4806801518e2cfe9e13229cfe06d8f \
    --hash=sha256:ae26d61dfa7a47befdc7572b521024e8745f3d809bd95ca9505a7bba9ef849ec \
    --hash=sha256:af8d94b0db561cf68b88a267c5c44b49e134f525d0dc2cb7ed413a66bc23559a \
    --hash=sha256:b343699e8308bdc51978310e1c959c584e7869cc8c40780058c87da7781a1e94 \
    --hash=sha256:b3c777e849237620b022f7f297dd67705f9f5cf1685f09f02e46f93e92725468 \
    --hash=sha256:b629de27fda84b42cde7edef0d85f13b958b47f6e9bbcbba9b673c562a89bd8b \
    --hash=sha256:ba09209fbe443b4acccebe845d8a138b89a8f4fbaeedd44953490b5315d5e965 \
    --hash=sha256:ba54cfebe86920a559a7c4d6b9050791c20513650a1952ebe3368c7dc70306f8 \
    --hash=sha256:bcb46e2f9feff8d06323983bd83ed00c201fdcab3d74973e7072a889b3979fcd \
    --hash=sha256:bcc33feacfaefce60c12fd500a277533bdc02b10a19f7f6d348763d8140bbba7 \
    --hash=sha256:bf16ba1b4d0b6b7c8e534936632270cf70eb00dbe09005bc345b2677b726855c \
    --hash=sha256:cf1845d02ad822a369a49f2bb9345b1614744267682e7a03527dc3bf6eea1777 \
    --hash=sha256:d69141514cc30b774ceea5e3ed3a6635c8d8a96edf664689b890f4089111fb35 \
    --hash=sha256:d9c7f76c0673154f044e9d78c8655fb4213f6ca31a836df48b40fe5d187717b9 \
    --hash=sha256:dbce0b29841537a2fa4a214c2bbf14de3587c9680caa9b4e217568472490b28f \
    --hash=sha256:dc624f6bc473dacdf7ef7eb8678d0d08edf15cd94fad6ae5c7d6cc67a4e4902f \
    --hash=sha256:e158cb00350dc278f3b91551101aa7d12415a66ebf2c91d8d5ac14e56ddd3ad0 \
    --hash=sha256:e491916b378fba47242221bb9ead245211b70d504f495d105d17b14a24b4907c \
    --hash=sha256:e795b7eb908249c4e43c7c99fac7c2c75dab0c43566e37db472a355f63693d71 \
    --hash=sha256:e7e480451b9fa137494bccd3a7d69adbe8ac65a87d97be61e11f1b1050a5bac3 \
    --hash=sha256:e91206ee562682b51b98ef4b26a6ef48fd84e15fd4c4bc5ec768eb641d206838 \
    --hash=sha256:e9871b1ffbfa9656b60aeee92ed5136a5742696006fa322b29ea3d8da0ecc9cf \
    --hash=sha256:e9aeb04d6aef139de265b29683e119b638208f88cf73cdd1658aa07221165321 \
    --hash=sha256:ebaea975e03d3141d9d3a507df75c9b3ec90fa9d2ffd07567b3a978d9d790b26 \
    --hash=sha256:f0606c8bf2cdefea14a43530f7657cbbb7ecf1c4222512492ef4a4434a9501ec \
    --hash=sha256:f13c32a3abd6079a66d9526e18dad9b6d280384d49d7c54040cd57b6424041d9 \
    --hash=sha256:f7401aebd7f581d7f83a439d87d474999317ee099218e5ad25d125290990ba65 \
    --hash=sha256:fa4ecea169a355be7a3ade2c783e2ed12f0e40d2c5621cda8b3297faf7fbb9f5 \
    --hash=sha256:fbd139c8447d25dd750ab79ee274cc5e1fe80fc56340ab10b18a195e1b6eca3e \
    --hash=sha256:fdafc9cce40277e0f7a0feabce0ee50dd2fa1800f3b38015e51296b5e814048d \
    --hash=sha256:fe3cca2e4e8a592be0f269a1ca4835c25199d9f3ce815c8491048f785b0a0198 \
    --hash=sha256:ffd0c5368496f41b0944be820fcb7a838aa6e623d250b01acf2643939c3f99d7
    # via torchvision
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via
    #   huggingface-hub
    #   transformers
regex==2026.9.29 \
    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \
    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \
    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \
    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \
    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \
    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \
    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \
    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \
    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \
    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \
    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \
    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \
    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \
    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \
    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \
    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \
    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \
    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \
    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \
    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \
    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \
    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \
    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \
    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \
    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \
    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \
    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \
    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \
    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \
    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \
    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \
    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \
    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \
    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \
    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \
    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \
    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \
    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \
    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \
    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \
    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \
    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \
    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \
    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \
    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \
    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \
    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \
    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \
    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \
    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \
    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \
    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \
    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \
    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \
    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \
    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \
    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \
    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \
    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \
    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \
    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \
    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \
    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \
    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \
    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \
    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \
    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \
    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \
    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \
    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \
    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \
    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \
    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \
    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \
    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \
    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \
    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \
    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \
    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \
    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \
    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \
    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \
    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \
    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \
    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \
    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \
    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \
    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \
    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \
    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \
    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \
    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \
    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \
    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \
    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \
    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \
    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \
    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \
    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \
    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \
    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \
    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \
    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \
    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \
    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \
    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \
    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \
    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \
    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \
    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \
    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \
    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \
    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \
    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \
    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \
    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \
    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \
    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \
    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \
    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \
    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \
    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \
    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \
    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \
    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \
    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \
    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \
    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \
    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \
    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52
    # via transformers
requests==2.34.2 \
    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \
    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed
    # via
    #   huggingface-hub
    #   transformers
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via
    #   -r tutorials/requirements-semantic-search-workshop.in
    #   transformers
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via torch
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via torch
tokenizers==0.22.2 \
    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \
    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \
    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \
    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \
    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \
    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \
    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \
    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \
    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \
    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \
    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \
    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \
    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \
    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \
    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \
    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \
    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \
    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \
    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \
    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \
    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \
    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \
    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \
    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5
    # via transformers
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via
    #   -r tutorials/requirements-semantic-search-workshop.in
    #   torchvision
torchaudio==2.11.0 \
    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \
    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \
    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \
    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \
    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \
    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \
    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \
    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \
    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \
    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \
    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \
    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \
    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \
    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \
    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \
    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \
    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \
    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \
    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \
    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \
    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \
    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \
    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \
    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \
    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \
    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \
    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \
    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84
    # via -r tutorials/requirements-semantic-search-workshop.in
torchvision==0.29.0 \
    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \
    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \
    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \
    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \
    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \
    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \
    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \
    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \
    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \
    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \
    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \
    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \
    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \
    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \
    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \
    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \
    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \
    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \
    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \
    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \
    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \
    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \
    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \
    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e
    # via -r tutorials/requirements-semantic-search-workshop.in
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via
    #   huggingface-hub
    #   transformers
transformers==4.57.6 \
    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \
    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3
    # via -r tutorials/requirements-semantic-search-workshop.in
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via torch
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   huggingface-hub
    #   torch
urllib3==2.8.0 \
    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \
    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63
    # via requests
'''

SKIP_INSTALL = os.environ.get("DIMER_NOTEBOOK_CI_PREINSTALLED") == "1"
ISOLATED_ENV = Path(os.environ.get("DIMER_ISOLATED_ENV", "dimer_isolated_env")).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / "bin" / "python"
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + "_tools")

if SKIP_INSTALL:
    print("DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.")
else:
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.")
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode("utf-8")).hexdigest() != LOCK_SHA256:
        raise RuntimeError("The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.")
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding="utf-8", newline="\n")
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError("The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.")
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith(".data/scripts/uv"))
        uv = ISOLATED_TOOLS / "uv"
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ)
    for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP"):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), "venv", "--quiet", "--managed-python", "--python", MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), "-c", "import platform; print(platform.python_version())"], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f"{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.")
    subprocess.run([str(uv), "pip", "install", "--quiet", "--python", str(ISOLATED_PYTHON), "--require-hashes", "--only-binary", ":all:", "--index-url", "https://pypi.org/simple", "-r", str(lock_path)], env=uv_env, check=True)
    print({"isolated_environment": str(ISOLATED_ENV), "isolated_python": isolated_version, "kernel_python": platform.python_version(), "locked_packages": LOCKED_PACKAGES, "setup_seconds": round(time.perf_counter() - setup_started)})

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    try:
        import google
    except ImportError:
        google = types.ModuleType("google")
        google.__path__ = []
        sys.modules["google"] = google
    _colab = types.ModuleType("google.colab")
    _files = types.ModuleType("google.colab.files")
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]

# Section 2 ran in this kernel before the worker existed: copy its settings into the worker's namespace.
CONFIGURATION_NAMES = (
    "USE_BYOD", "DOCUMENTS_PATH", "QUERIES_PATH", "BYOD_EMBEDDING_INSTRUCTION", "BYOD_RERANK_INSTRUCTION",
    "SAMPLE_SEED", "RERANK_K", "EMBEDDING_INSTRUCTION", "RERANK_INSTRUCTION", "RUN_K_SWEEP", "K_SWEEP_VALUES",
    "OUTPUT_DIR",
)

# Imports the model libraries and records the runtime inventory. It runs in the isolated environment.
RUNTIME_INVENTORY = r'''
import sys

import numpy as np
import torch
import transformers
import huggingface_hub
import safetensors

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE.startswith("cuda") else torch.float32

RUNTIME = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "numpy": np.__version__,
    "numpy_source": "pinned install in the interpreter named by environment",
    "environment": sys.prefix,
    "device": DEVICE,
    "dtype": str(DTYPE),
    "cuda_available": torch.cuda.is_available(),
}
print(RUNTIME)
'''

if SKIP_INSTALL:
    print("Routing disabled: the notebook runs in this kernel.")
    exec(RUNTIME_INVENTORY)
else:
    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [
        t for t in _ip.input_transformers_cleanup if getattr(t, "__name__", "") != "_route_to_isolated_runtime"
    ]
    if isinstance(globals().get("_DIMER_ISOLATED_RUNTIME"), IsolatedRuntime):
        _DIMER_ISOLATED_RUNTIME.close()
    _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print(f"Every later code cell now runs in {ISOLATED_PYTHON} (pid {_DIMER_ISOLATED_RUNTIME.proc.pid}).")
    _DIMER_ISOLATED_RUNTIME.run("".join(f"{name} = {globals()[name]!r}\n" for name in CONFIGURATION_NAMES))
    _DIMER_ISOLATED_RUNTIME.run(RUNTIME_INVENTORY)

{'isolated_environment': '/content/dimer_isolated_env', 'isolated_python': '3.12.12', 'kernel_python': '3.13.15', 'locked_packages': 47, 'setup_seconds': 65}
Every later code cell now runs in /content/dimer_isolated_env/bin/python (pid 3829).
{'python': '3.12.12', 'torch': '2.14.0+cu130', 'transformers': '4.57.6', 'huggingface_hub': '0.36.2', 'numpy': '2.1.3', 'numpy_source': 'pinned install in the interpreter named by environment', 'environment': '/content/dimer_isolated_env', 'device': 'cuda:0', 'dtype': 'torch.bfloat16', 'cuda_available': True}


## 4. Immutable model provenance and snapshot verification

The notebook embeds the same immutable snapshot manifests used by the live DIMER carriers.

For each model it:

1. writes the trusted manifest locally;
2. downloads only manifest-listed files from the exact immutable Hugging Face revision;
3. checks byte size and SHA-256 for every listed file; and
4. loads only from the verified local snapshot with `trust_remote_code=False`.

A verification failure is terminal. The notebook does not fall back to another checkpoint or a mutable branch.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [3]:
import hashlib
import json
from pathlib import Path
from huggingface_hub import hf_hub_download

EMBED_MANIFEST = json.loads(r"""{
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "qwen3-embedding-0.6b",
  "modelId": "Qwen/Qwen3-Embedding-0.6B",
  "revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",
  "files": [
    {
      "path": "1_Pooling/config.json",
      "bytes": 313,
      "sha256": "37bf193fa101f19101bfad9c31d3eb0f786e247b7b1e5cb7f007d730eed1ddbd"
    },
    {
      "path": "README.md",
      "bytes": 17237,
      "sha256": "c34d9b7e5a267ad3fdd13227a253686bc90844ff4744a2a6a86c7c905e3d06f3"
    },
    {
      "path": "config.json",
      "bytes": 727,
      "sha256": "b5bf1f51fc45be473a54718cef92448d90a1be001bf9b9a44b8c7f10a19feaa9"
    },
    {
      "path": "config_sentence_transformers.json",
      "bytes": 215,
      "sha256": "10667c72ddb772627bf1780cb7f86af8e2ae0032b8c243c731172064105c6961"
    },
    {
      "path": "generation_config.json",
      "bytes": 117,
      "sha256": "28396d421a2108acce96383f6a7de78008f7f1b17f807958f3c14c51dbfb65fb"
    },
    {
      "path": "merges.txt",
      "bytes": 1671853,
      "sha256": "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1191586416,
      "sha256": "0437e45c94563b09e13cb7a64478fc406947a93cb34a7e05870fc8dcd48e23fd"
    },
    {
      "path": "modules.json",
      "bytes": 349,
      "sha256": "84e40c8e006c9b1d6c122e02cba9b02458120b5fb0c87b746c41e0207cf642cf"
    },
    {
      "path": "tokenizer.json",
      "bytes": 11423705,
      "sha256": "def76fb086971c7867b829c23a26261e38d9d74e02139253b38aeb9df8b4b50a"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 9706,
      "sha256": "253153d0738ceb4c668d2eff957714dd2bea0b56de772a9fdccd96cbf517e6a0"
    },
    {
      "path": "vocab.json",
      "bytes": 2776833,
      "sha256": "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910"
    }
  ],
  "totalBytes": 1207487471
}""")
RERANK_MANIFEST = json.loads(r"""{
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "qwen3-reranker-0.6b",
  "modelId": "Qwen/Qwen3-Reranker-0.6B",
  "revision": "e61197ed45024b0ed8a2d74b80b4d909f1255473",
  "files": [
    {
      "path": "1_LogitScore/config.json",
      "bytes": 57,
      "sha256": "73e3156450564d8a98b7e47bcf5aace0f29600828b51937da545571e84db3ff3"
    },
    {
      "path": "README.md",
      "bytes": 14742,
      "sha256": "5bba8c734f6dd3ae48317b4139317e45a7fce48fc55e15670b23a0dd15492ab6"
    },
    {
      "path": "chat_template.jinja",
      "bytes": 741,
      "sha256": "6f682162495ec5b39fd9005c01b6aa2a74669379fe967039f1e2cbbe8752369d"
    },
    {
      "path": "config.json",
      "bytes": 727,
      "sha256": "d479c427a9ca5295218063d4f9aca4f297ab4ac27487cca7af42c84643d51ef0"
    },
    {
      "path": "config_sentence_transformers.json",
      "bytes": 325,
      "sha256": "6a153d6696f78fd588c1c728967f0b773ea869d3c6028f151ce71ebe49140762"
    },
    {
      "path": "generation_config.json",
      "bytes": 214,
      "sha256": "81051cd3f6e77013827148d0b8a6ead93f8ac390d5ab805f849199f0af6a08db"
    },
    {
      "path": "merges.txt",
      "bytes": 1671853,
      "sha256": "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1191588280,
      "sha256": "27cd75a405b9c1b46b59abfd88aaa209e6fed2a1972cde9b70e7659537c5e65b"
    },
    {
      "path": "modules.json",
      "bytes": 280,
      "sha256": "6f13b6b4a89e577b591b2077bca40c67c26541a6740a8809267cb474f90806a9"
    },
    {
      "path": "sentence_bert_config.json",
      "bytes": 362,
      "sha256": "3234ebd224d492cbe8d55d5ec80a3f408451c4db3005bafb64fe1c51c763e01e"
    },
    {
      "path": "tokenizer.json",
      "bytes": 11422654,
      "sha256": "aeb13307a71acd8fe81861d94ad54ab689df773318809eed3cbe794b4492dae4"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 9706,
      "sha256": "253153d0738ceb4c668d2eff957714dd2bea0b56de772a9fdccd96cbf517e6a0"
    },
    {
      "path": "vocab.json",
      "bytes": 2776833,
      "sha256": "ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910"
    }
  ],
  "totalBytes": 1207486774
}""")

EMBED_DIR = Path("weights/qwen3-embedding-0.6b")
RERANK_DIR = Path("weights/qwen3-reranker-0.6b")
MANIFEST_NAME = "dimer-base-manifest.json"

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_manifest(root, manifest):
    root.mkdir(parents=True, exist_ok=True)
    path = root / MANIFEST_NAME
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return path

def stage_and_verify_snapshot(root, manifest):
    write_manifest(root, manifest)
    model_id = manifest["modelId"]
    revision = manifest["revision"]

    fetched = []
    for entry in manifest["files"]:
        target = root / entry["path"]
        if not target.is_file():
            target.parent.mkdir(parents=True, exist_ok=True)
            hf_hub_download(
                repo_id=model_id,
                filename=entry["path"],
                revision=revision,
                local_dir=str(root),
            )
            fetched.append(entry["path"])

    for entry in manifest["files"]:
        path = root / entry["path"]
        if not path.is_file():
            raise FileNotFoundError(f"Snapshot file missing after staging: {path}")
        actual_size = path.stat().st_size
        if actual_size != entry["bytes"]:
            raise ValueError(
                f"{model_id} {entry['path']} size {actual_size} != {entry['bytes']}"
            )
        actual_sha = sha256_file(path)
        if actual_sha != entry["sha256"]:
            raise ValueError(
                f"{model_id} {entry['path']} sha256 {actual_sha} != {entry['sha256']}"
            )
    return {
        "model_id": model_id,
        "revision": revision,
        "files": len(manifest["files"]),
        "bytes": manifest["totalBytes"],
        "fetched": fetched,
    }

embed_snapshot = stage_and_verify_snapshot(EMBED_DIR, EMBED_MANIFEST)
rerank_snapshot = stage_and_verify_snapshot(RERANK_DIR, RERANK_MANIFEST)

print("Embedding snapshot:", embed_snapshot)
print("Reranker snapshot:", rerank_snapshot)

Embedding snapshot: {'model_id': 'Qwen/Qwen3-Embedding-0.6B', 'revision': '97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3', 'files': 11, 'bytes': 1207487471, 'fetched': ['1_Pooling/config.json', 'README.md', 'config.json', 'config_sentence_transformers.json', 'generation_config.json', 'merges.txt', 'model.safetensors', 'modules.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']}
Reranker snapshot: {'model_id': 'Qwen/Qwen3-Reranker-0.6B', 'revision': 'e61197ed45024b0ed8a2d74b80b4d909f1255473', 'files': 13, 'bytes': 1207486774, 'fetched': ['1_LogitScore/config.json', 'README.md', 'chat_template.jinja', 'config.json', 'config_sentence_transformers.json', 'generation_config.json', 'merges.txt', 'model.safetensors', 'modules.json', 'sentence_bert_config.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']}


## 5. Dataset provenance

The default sample uses **Banking77** (Casanueva et al., 2020; CC BY 4.0), fetched from a pinned commit of the PolyAI task-specific datasets repository.

The source contains 13,083 customer-support messages labelled with 77 fine-grained banking intents. In this notebook:

- each intent identifier becomes a short candidate **document** such as `cash withdrawal`;
- the corpus therefore contains exactly **77 candidate documents**;
- the evaluation set contains **154 messages from Banking77's official test partition** — exactly two queries per intent;
- there is no training, adaptation, validation-based selection, or model selection in this notebook.

This is an intentionally small and interpretable retrieval surrogate, not a simulation of large enterprise document search. Pretraining overlap with Banking77 cannot be ruled out.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [4]:
import csv
import io
import random
import urllib.request
from collections import defaultdict

CORPUS_NAME = "Banking77"
CORPUS_LICENSE = "CC BY 4.0 (Casanueva et al. 2020; PolyAI-LDN/task-specific-datasets)"
CORPUS_RELEASE = "PolyAI-LDN/task-specific-datasets @ 57ec275d8078af65b7731c2a98be812d844a6d6b"
CORPUS_BASE_URL = (
    "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/"
    "57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/"
)
CORPUS_FILES = {
    "train": ("train.csv", 839_073, "b06e26ac675513959a63135f11b94ea7786ed02da65db93a5650d8838cbc664b"),
    "test": ("test.csv", 239_961, "d12d6e3bc4c3103966ae786dc435913c0c563dfa328f5a3646d0e62cfeeb474d"),
}
CORPUS_ROWS = {"train": 10_003, "test": 3_080}
CORPUS_INTENTS = 77
CORPUS_CACHE = Path("weights/banking77")
MAX_TEXT_CHARS = 100_000

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def fetch_corpus_file(split):
    name, expected_size, expected_sha = CORPUS_FILES[split]
    CORPUS_CACHE.mkdir(parents=True, exist_ok=True)
    path = CORPUS_CACHE / name

    data = path.read_bytes() if path.is_file() else b""
    if len(data) != expected_size or sha256_bytes(data) != expected_sha:
        with urllib.request.urlopen(CORPUS_BASE_URL + name, timeout=120) as response:
            data = response.read()
        if len(data) != expected_size or sha256_bytes(data) != expected_sha:
            raise ValueError(f"{name}: pinned size/digest verification failed")
        path.write_bytes(data)
    return data

def read_corpus_csv(data, split):
    rows = list(csv.DictReader(io.StringIO(data.decode("utf-8"))))
    if len(rows) != CORPUS_ROWS[split]:
        raise ValueError(f"{split}: {len(rows)} rows != expected {CORPUS_ROWS[split]}")
    if not rows or not {"text", "category"}.issubset(rows[0]):
        raise ValueError(f"{split}: expected columns text and category")
    cleaned = []
    for i, row in enumerate(rows):
        text = row["text"].strip()
        intent = row["category"].strip()
        if not text or not intent:
            continue
        if len(text) > MAX_TEXT_CHARS:
            continue
        cleaned.append({"source_id": f"{split}-{i:05d}", "text": text, "intent": intent})
    intents = {r["intent"] for r in cleaned}
    if len(intents) != CORPUS_INTENTS:
        raise ValueError(f"{split}: {len(intents)} intents != expected {CORPUS_INTENTS}")
    return cleaned

train_rows = read_corpus_csv(fetch_corpus_file("train"), "train")
test_rows = read_corpus_csv(fetch_corpus_file("test"), "test")

print({
    "corpus": CORPUS_NAME,
    "release": CORPUS_RELEASE,
    "license": CORPUS_LICENSE,
    "train_rows": len(train_rows),
    "test_rows": len(test_rows),
    "intents": len({r["intent"] for r in train_rows}),
})

{'corpus': 'Banking77', 'release': 'PolyAI-LDN/task-specific-datasets @ 57ec275d8078af65b7731c2a98be812d844a6d6b', 'license': 'CC BY 4.0 (Casanueva et al. 2020; PolyAI-LDN/task-specific-datasets)', 'train_rows': 10003, 'test_rows': 3080, 'intents': 77}


## 6. Build and validate the retrieval problem

Every intent becomes one document. The evaluation queries come only from the official test split.

Validation happens before either model runs:

- 77 unique intent documents;
- unique document IDs and unique normalized document texts;
- 154 unique held-out queries;
- exactly two queries per intent;
- every gold document exists in the corpus;
- no empty or over-limit text.

In [5]:
def intent_phrase(intent):
    return " ".join(intent.strip().split("_"))

intents = sorted({r["intent"] for r in train_rows})
documents = [
    {
        "doc_id": f"intent_{i:03d}",
        "text": intent_phrase(intent),
        "intent": intent,
    }
    for i, intent in enumerate(intents)
]
intent_to_doc = {d["intent"]: d["doc_id"] for d in documents}
doc_id_to_index = {d["doc_id"]: i for i, d in enumerate(documents)}

by_intent = defaultdict(list)
for row in test_rows:
    by_intent[row["intent"]].append(row)

rng = random.Random(SAMPLE_SEED)
queries = []
for intent in intents:
    pool = list(by_intent[intent])
    rng.shuffle(pool)
    if len(pool) < 2:
        raise ValueError(f"Intent {intent!r} has fewer than two test examples")
    for source in pool[:2]:
        queries.append({
            "query_id": source["source_id"],
            "query": source["text"],
            "gold_doc_id": intent_to_doc[intent],
            "intent": intent,
        })
rng.shuffle(queries)

def validate_problem(documents, queries):
    if len(documents) != 77:
        raise ValueError(f"Expected 77 documents, got {len(documents)}")
    if len(queries) != 154:
        raise ValueError(f"Expected 154 queries, got {len(queries)}")
    doc_ids = [d["doc_id"] for d in documents]
    doc_texts = [d["text"].strip().lower() for d in documents]
    if len(set(doc_ids)) != len(doc_ids):
        raise ValueError("Duplicate doc_id")
    if len(set(doc_texts)) != len(doc_texts):
        raise ValueError("Duplicate normalized document text")
    query_ids = [q["query_id"] for q in queries]
    if len(set(query_ids)) != len(query_ids):
        raise ValueError("Duplicate query_id")
    counts = defaultdict(int)
    for d in documents:
        if not d["text"].strip() or len(d["text"]) > MAX_TEXT_CHARS:
            raise ValueError(f"Invalid document {d['doc_id']}")
    for q in queries:
        if not q["query"].strip() or len(q["query"]) > MAX_TEXT_CHARS:
            raise ValueError(f"Invalid query {q['query_id']}")
        if q["gold_doc_id"] not in doc_id_to_index:
            raise ValueError(f"Missing gold document for {q['query_id']}")
        counts[q["intent"]] += 1
    if set(counts.values()) != {2} or len(counts) != 77:
        raise ValueError("Evaluation set must contain exactly two queries per intent")
    payload = json.dumps(
        {"documents": documents, "queries": queries},
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return {
        "documents": len(documents),
        "queries": len(queries),
        "queries_per_intent": 2,
        "digest": hashlib.sha256(payload).hexdigest(),
    }

problem_manifest = validate_problem(documents, queries)
print(problem_manifest)
print("Example documents:", documents[:5])
print("Example query:", queries[0])

{'documents': 77, 'queries': 154, 'queries_per_intent': 2, 'digest': '35e065e5503a91d2ee4b411414263e4a204331ceebb43ab741a2587d75578e26'}
Example documents: [{'doc_id': 'intent_000', 'text': 'Refund not showing up', 'intent': 'Refund_not_showing_up'}, {'doc_id': 'intent_001', 'text': 'activate my card', 'intent': 'activate_my_card'}, {'doc_id': 'intent_002', 'text': 'age limit', 'intent': 'age_limit'}, {'doc_id': 'intent_003', 'text': 'apple pay or google pay', 'intent': 'apple_pay_or_google_pay'}, {'doc_id': 'intent_004', 'text': 'atm support', 'intent': 'atm_support'}]
Example query: {'query_id': 'test-02331', 'query': 'I would like to transfer some money from my other bank account into this one.', 'gold_doc_id': 'intent_065', 'intent': 'transfer_into_account'}


## 7. Non-neural baselines

Before using either model, we establish two reference points:

- **Random floor:** the expected performance of a uniformly random ordering of 77 documents.
- **Lexical baseline:** Jaccard overlap between lower-cased alphanumeric query/document tokens.

The lexical baseline is intentionally simple. It provides a no-model comparison without introducing another retrieval package.

**Tie policy.** Every system is scored on the ranking the notebook displays and exports: documents are sorted by descending score, and exact ties keep document order (the dense retriever's stable sort follows the same rule). Jaccard scores tie often — many queries share no word with most intent phrases — so the lexical cell also reports a named diagnostic in which every tied document is ranked above the gold one. That conservative figure is the policy earlier versions of this notebook reported as the lexical result; compare the two to see how much of the lexical score depends on tie order.

> **Before you run it:** predict whether this simple reference will be easy or difficult for the learned model(s) to beat. After the result appears, record the baseline value before looking at the more complex model comparison.

In [6]:
import math
import re
import statistics

TOKEN_RE = re.compile(r"[a-z0-9]+")

def tokens(text):
    return set(TOKEN_RE.findall(text.lower()))

def jaccard(a, b):
    x, y = tokens(a), tokens(b)
    if not x or not y:
        return 0.0
    return len(x & y) / len(x | y)

def pessimistic_rank(scores, positive_index):
    target = float(scores[positive_index])
    return 1 + sum(
        1 for i, score in enumerate(scores)
        if i != positive_index and float(score) >= target
    )

def stable_order(scores):
    """Descending score; exact ties keep document order (the same rule as the dense retriever's stable argsort)."""
    return sorted(range(len(scores)), key=lambda i: (-float(scores[i]), i))

def attention_budget_batches(lengths, max_items, max_cost):
    """Consecutive (start, end) batches of at most `max_items` inputs whose attention size, inputs x longest^2
    tokens, stays within `max_cost`. Short inputs keep fixed-size batches; long inputs get smaller batches."""
    batches, start, longest = [], 0, 0
    for i, n in enumerate(lengths):
        size, grown = i - start, max(longest, n)
        if size and (size + 1 > max_items or (size + 1) * grown * grown > max_cost):
            batches.append((start, i))
            start, longest = i, n
        else:
            longest = grown
    if lengths:
        batches.append((start, len(lengths)))
    return batches

def metrics_from_ranks(ranks, n_documents, ks=(1, 3, 6, 10)):
    if not ranks:
        raise ValueError("No ranks to score")
    result = {
        "n_queries": len(ranks),
        "n_documents": n_documents,
        "mrr": sum(1.0 / r for r in ranks) / len(ranks),
        "median_rank": float(statistics.median(ranks)),
    }
    for k in ks:
        result[f"recall@{k}"] = sum(r <= k for r in ranks) / len(ranks)
    return result

def composed_metrics(final_ranks, depth, ks=(1, 3, 6)):
    """Full-system metrics for a reranked top-`depth` list; None marks a gold document outside the shortlist.

    Each recall@k is computed at its literal cutoff. A cutoff deeper than the list the composed system returns
    is reported as None (not measurable) rather than relabelled recall@depth.
    """
    if not final_ranks:
        raise ValueError("No queries to score")
    n = len(final_ranks)
    result = {"n_queries": n, "rerank_k": depth}
    for k in sorted(set(ks) | {depth}):
        result[f"recall@{k}"] = None if k > depth else sum(r is not None and r <= k for r in final_ranks) / n
    result["mrr"] = sum(0.0 if r is None else 1.0 / r for r in final_ranks) / n
    return result

def conditional_reranker_metrics(covered_ranks, depth):
    """Reranker quality on queries whose gold document entered the shortlist; explicit when none did."""
    ks = tuple(k for k in sorted({1, 3, depth}) if k <= depth)
    if not covered_ranks:
        return {
            "n_queries": 0,
            "n_documents": depth,
            "applicable": False,
            "reason": "no gold document entered any shortlist",
            "mrr": None,
            "median_rank": None,
            **{f"recall@{k}": None for k in ks},
        }
    return metrics_from_ranks(covered_ranks, depth, ks=ks)

def random_floor(n_documents, ks=(1, 3, 6, 10)):
    result = {
        "n_documents": n_documents,
        "mrr": sum(1.0 / r for r in range(1, n_documents + 1)) / n_documents,
    }
    for k in ks:
        result[f"recall@{k}"] = min(k, n_documents) / n_documents
    return result

# Shared output checks: the canonical path, the K-sweep and BYOD all call these before using model outputs.
def check_embedding_matrix(vectors, expected_rows, expected_dim, name, atol=2e-4):
    arr = np.asarray(vectors)
    if arr.shape != (expected_rows, expected_dim):
        raise RuntimeError(f"{name}: embedding shape {arr.shape} != ({expected_rows}, {expected_dim})")
    if not np.isfinite(arr).all():
        raise RuntimeError(f"{name}: embeddings contain non-finite values")
    norms = np.linalg.norm(arr, axis=1)
    if not np.allclose(norms, 1.0, atol=atol):
        raise RuntimeError(f"{name}: embeddings are not unit norm: range {norms.min()}..{norms.max()}")
    return norms

def check_relevance_scores(scores, expected_count, name):
    values = [float(s) for s in scores]
    if len(values) != expected_count:
        raise RuntimeError(f"{name}: reranker returned {len(values)} scores for {expected_count} pairs")
    bad = [i for i, s in enumerate(values) if not (math.isfinite(s) and 0.0 <= s <= 1.0)]
    if bad:
        raise RuntimeError(f"{name}: non-finite or out-of-range reranker scores at pair positions {bad[:5]}")
    return values

def check_permutation(order, n_items, name):
    positions = [int(i) for i in order]
    if sorted(positions) != list(range(n_items)):
        raise RuntimeError(f"{name}: ranking is not a permutation of the {n_items} candidates")
    return positions

def check_reranked_group(candidate_ids, reranked_rows, name):
    ids = [row["doc_id"] for row in reranked_rows]
    if len(set(candidate_ids)) != len(candidate_ids) or sorted(ids) != sorted(candidate_ids):
        raise RuntimeError(f"{name}: reranked IDs are not exactly the unique shortlist IDs")
    if [row["reranked_rank"] for row in reranked_rows] != list(range(1, len(ids) + 1)):
        raise RuntimeError(f"{name}: reranked ranks are not 1..{len(ids)}")

# Tie policy (the same for every system): a gold document's rank is its position in the ranking the
# notebook displays and exports. Jaccard scores tie often, so the historical "ties counted against gold"
# (pessimistic) rank is also reported as a named diagnostic.
lexical_ranks = []
lexical_pessimistic_ranks = []
lexical_gold_tied = 0
lexical_rows = []
for q in queries:
    scores = [jaccard(q["query"], d["text"]) for d in documents]
    gold_index = doc_id_to_index[q["gold_doc_id"]]
    order = stable_order(scores)
    lexical_ranks.append(order.index(gold_index) + 1)
    lexical_pessimistic_ranks.append(pessimistic_rank(scores, gold_index))
    lexical_gold_tied += any(i != gold_index and scores[i] == scores[gold_index] for i in range(len(scores)))
    for r, idx in enumerate(order, start=1):
        lexical_rows.append({
            "query_id": q["query_id"],
            "doc_id": documents[idx]["doc_id"],
            "rank": r,
            "score": float(scores[idx]),
            "is_gold": documents[idx]["doc_id"] == q["gold_doc_id"],
        })

METRIC_KS = tuple(sorted({1, 3, 6, 10, RERANK_K}))
random_metrics = random_floor(len(documents), ks=METRIC_KS)
lexical_metrics = metrics_from_ranks(lexical_ranks, len(documents), ks=METRIC_KS)
lexical_metrics["tie_policy"] = "displayed ranking (exact ties keep document order)"
lexical_pessimistic_metrics = metrics_from_ranks(lexical_pessimistic_ranks, len(documents), ks=METRIC_KS)
lexical_pessimistic_metrics["tie_policy"] = "diagnostic: every tied document ranked above gold"
lexical_tie_diagnostic = {
    "queries_with_gold_score_tied": lexical_gold_tied,
    "fraction": lexical_gold_tied / len(queries),
}
print("Random floor:", random_metrics)
print("Lexical baseline:", lexical_metrics)
print("Lexical, ties counted against gold (diagnostic):", lexical_pessimistic_metrics)
print("Lexical tie diagnostic:", lexical_tie_diagnostic)

Random floor: {'n_documents': 77, 'mrr': 0.06399351348411561, 'recall@1': 0.012987012987012988, 'recall@3': 0.03896103896103896, 'recall@6': 0.07792207792207792, 'recall@10': 0.12987012987012986}
Lexical baseline: {'n_queries': 154, 'n_documents': 77, 'mrr': 0.4845958399149653, 'median_rank': 3.0, 'recall@1': 0.35064935064935066, 'recall@3': 0.5454545454545454, 'recall@6': 0.6883116883116883, 'recall@10': 0.7727272727272727, 'tie_policy': 'displayed ranking (exact ties keep document order)'}
Lexical, ties counted against gold (diagnostic): {'n_queries': 154, 'n_documents': 77, 'mrr': 0.4394609528490353, 'median_rank': 4.0, 'recall@1': 0.3116883116883117, 'recall@3': 0.487012987012987, 'recall@6': 0.6493506493506493, 'recall@10': 0.7402597402597403, 'tie_policy': 'diagnostic: every tied document ranked above gold'}
Lexical tie diagnostic: {'queries_with_gold_score_tied': 93, 'fraction': 0.6038961038961039}


## 8. Stage 1 — build the dense document index

Qwen3-Embedding uses:

- left padding;
- last-token pooling;
- 1024-dimensional representations;
- L2 normalization;
- an instruction prefix for **queries only**.

The 77 document vectors are the reusable offline index. Cosine similarity is implemented as a dot product because both query and document vectors are unit normalized.

Cosine similarity is an ordering signal, not a calibrated probability.

**Core concept:** an embedding is a list of learned numerical features. L2 normalization gives each vector unit length; cosine then measures direction rather than magnitude. Last-token pooling takes the final non-padding token’s representation. **Expected result:** 77 rows × 1024 features, finite values and norms near one; those checks must pass before searching.

In [7]:
import time
from transformers import AutoModel, AutoTokenizer

EMBED_MODEL_ID = EMBED_MANIFEST["modelId"]
EMBED_MODEL_REVISION = EMBED_MANIFEST["revision"]
EMBED_DIM = 1024
EMBED_MAX_TOKENS = 8192
EMBED_MAX_BATCH = 64
# Attention memory grows with inputs x length^2. Cap each batch at one full-length input's worth; the canonical
# short texts still run in full 64-item batches, while long BYOD inputs are processed a few at a time.
EMBED_BATCH_ATTENTION = EMBED_MAX_TOKENS ** 2

class EmbeddingRuntime:
    def __init__(self, root, device=DEVICE):
        self.device = device
        self.dtype = torch.bfloat16 if device.startswith("cuda") else torch.float32
        self.tokenizer = AutoTokenizer.from_pretrained(
            str(root),
            padding_side="left",
            trust_remote_code=False,
            local_files_only=True,
        )
        self.model = AutoModel.from_pretrained(
            str(root),
            dtype=self.dtype,
            trust_remote_code=False,
            local_files_only=True,
        ).to(device).eval()

    @staticmethod
    def format_query(query, instruction):
        return f"Instruct: {instruction}\nQuery:{query}"

    def embed_batch(self, texts, kind, instruction):
        if kind not in {"query", "document"}:
            raise ValueError("kind must be query or document")
        if not 1 <= len(texts) <= EMBED_MAX_BATCH:
            raise ValueError(f"Embedding batch must contain 1..{EMBED_MAX_BATCH} texts")
        for i, text in enumerate(texts):
            if not isinstance(text, str) or not text.strip():
                raise ValueError(f"texts[{i}] must be a non-empty string")
            if len(text) > MAX_TEXT_CHARS:
                raise ValueError(f"texts[{i}] exceeds {MAX_TEXT_CHARS} characters")

        formatted = [
            self.format_query(text, instruction) if kind == "query" else text
            for text in texts
        ]
        batch = self.tokenizer(
            formatted,
            padding=True,
            truncation=True,
            max_length=EMBED_MAX_TOKENS,
            return_tensors="pt",
        ).to(self.device)

        with torch.inference_mode():
            hidden = self.model(**batch).last_hidden_state
        pooled = hidden[:, -1].float()
        normalized = torch.nn.functional.normalize(pooled, dim=-1)
        counts = batch["attention_mask"].sum(dim=1).tolist()
        return normalized.cpu().numpy().astype(np.float32), [int(x) for x in counts]

    def token_lengths(self, texts, kind, instruction):
        """Token counts before truncation (instruction prompt included), for the truncation audit."""
        formatted = [
            self.format_query(text, instruction) if kind == "query" else text
            for text in texts
        ]
        encoded = self.tokenizer(formatted, padding=False, truncation=False, return_attention_mask=False)
        return [len(ids) for ids in encoded["input_ids"]]

    def embed_all(self, texts, kind, instruction):
        lengths = [min(n, EMBED_MAX_TOKENS) for n in self.token_lengths(texts, kind, instruction)]
        vectors, counts = [], []
        for start, end in attention_budget_batches(lengths, EMBED_MAX_BATCH, EMBED_BATCH_ATTENTION):
            v, c = self.embed_batch(texts[start:end], kind, instruction)
            vectors.append(v)
            counts.extend(c)
        return np.vstack(vectors), counts

t0 = time.perf_counter()
embedder = EmbeddingRuntime(EMBED_DIR)

doc_texts = [d["text"] for d in documents]
document_vectors, document_token_counts = embedder.embed_all(
    doc_texts, "document", EMBEDDING_INSTRUCTION
)
index_seconds = time.perf_counter() - t0

norms = check_embedding_matrix(document_vectors, len(documents), EMBED_DIM, "Document index")

print({
    "n_documents": len(documents),
    "embedding_dim": document_vectors.shape[1],
    "index_build_seconds": round(index_seconds, 3),
    "device": DEVICE,
    "dtype": str(DTYPE),
    "norm_range": [float(norms.min()), float(norms.max())],
})

{'n_documents': 77, 'embedding_dim': 1024, 'index_build_seconds': 11.317, 'device': 'cuda:0', 'dtype': 'torch.bfloat16', 'norm_range': [0.9999998807907104, 1.0000001192092896]}


## 9. Retrieve candidates for every held-out query

We embed all 154 queries with the retrieval instruction and rank all 77 documents by cosine similarity.

The output of this stage is the complete first-stage ranking. We retain it separately from the later reranked shortlist so that the two stages can be evaluated independently.

**Predict before running:** which queries might lexical overlap miss but semantic retrieval find? Write one example from the sample and a reason. **What to notice:** the ranking covers all 77 documents; a gold rank of 1 contributes 1 to MRR, rank 4 contributes 1/4.

In [8]:
t0 = time.perf_counter()
query_vectors, query_token_counts = embedder.embed_all(
    [q["query"] for q in queries],
    "query",
    EMBEDDING_INSTRUCTION,
)
query_embed_seconds = time.perf_counter() - t0
check_embedding_matrix(query_vectors, len(queries), EMBED_DIM, "Query embeddings")

t0 = time.perf_counter()
similarities = query_vectors @ document_vectors.T
retrieval_seconds = time.perf_counter() - t0

retrieval_rankings = []
retrieval_rows = []
retriever_gold_ranks = []

for qi, q in enumerate(queries):
    scores = similarities[qi]
    order = np.argsort(-scores, kind="stable")
    check_permutation(order, len(documents), f"Retrieval ranking for {q['query_id']}")

    gold_index = doc_id_to_index[q["gold_doc_id"]]
    gold_rank = int(np.where(order == gold_index)[0][0]) + 1
    retriever_gold_ranks.append(gold_rank)

    ranking = []
    for rank, idx in enumerate(order, start=1):
        row = {
            "query_id": q["query_id"],
            "doc_id": documents[int(idx)]["doc_id"],
            "retrieval_rank": rank,
            "retrieval_score": float(scores[int(idx)]),
            "is_gold": documents[int(idx)]["doc_id"] == q["gold_doc_id"],
        }
        ranking.append(row)
        retrieval_rows.append(dict(row))
    retrieval_rankings.append(ranking)

retriever_metrics = metrics_from_ranks(
    retriever_gold_ranks, len(documents), ks=METRIC_KS
)

print("Retriever metrics:", retriever_metrics)
print({
    "query_embedding_seconds": round(query_embed_seconds, 3),
    "retrieval_matrix_seconds": round(retrieval_seconds, 6),
    "mean_query_embedding_seconds": round(query_embed_seconds / len(queries), 6),
})

Retriever metrics: {'n_queries': 154, 'n_documents': 77, 'mrr': 0.731011930735903, 'median_rank': 1.0, 'recall@1': 0.6103896103896104, 'recall@3': 0.8116883116883117, 'recall@6': 0.9155844155844156, 'recall@10': 0.9545454545454546}
{'query_embedding_seconds': 4.0, 'retrieval_matrix_seconds': 0.005776, 'mean_query_embedding_seconds': 0.025974}


## 10. Build the reranking shortlist

The default shortlist depth is `RERANK_K = 6`.

`shortlist_coverage@K` is the fraction of queries for which the gold document appears anywhere in the first-stage top-*K*. It is the **maximum possible end-to-end recall@1** for any reranker restricted to that shortlist.

If the gold document is missing here, the second stage has no opportunity to put it first.

**Checkpoint:** if shortlist coverage is 0.8, can end-to-end recall@1 be 0.9?

<details><summary>Sample answer</summary>
No. At least 20% of queries have already lost the relevant document. Even perfect ordering of every remaining shortlist cannot exceed 0.8. Coverage is a ceiling, not a promised outcome.
</details>

In [9]:
shortlists = []
coverage_flags = []

for q, ranking in zip(queries, retrieval_rankings, strict=True):
    topk = ranking[:RERANK_K]
    if len(topk) != RERANK_K:
        raise RuntimeError("Shortlist length mismatch")
    if len({row["doc_id"] for row in topk}) != RERANK_K:
        raise RuntimeError("Shortlist contains duplicate documents")

    present = any(row["is_gold"] for row in topk)
    coverage_flags.append(present)
    shortlists.append({
        "query_id": q["query_id"],
        "query": q["query"],
        "gold_doc_id": q["gold_doc_id"],
        "candidates": topk,
    })

shortlist_coverage = sum(coverage_flags) / len(coverage_flags)
print({
    "rerank_k": RERANK_K,
    "shortlist_coverage@k": shortlist_coverage,
    "covered_queries": sum(coverage_flags),
    "unrecoverable_queries": len(coverage_flags) - sum(coverage_flags),
})

{'rerank_k': 6, 'shortlist_coverage@k': 0.9155844155844156, 'covered_queries': 141, 'unrecoverable_queries': 13}


## 11. Prediction exercise

Inspect the first shortlist below before reading the later reranker output. Each candidate is printed with its intent phrase, so you can judge the candidates from their text as well as their IDs and scores.

Consider:

1. Is the gold document already present?
2. Is it currently at rank 1?
3. Which candidate phrases appear semantically close enough that a cross-encoder might reorder them?

The next stage will reveal what the reranker actually does. The exercise is interpretive and does not block `Run all`.

In [10]:
def preview(text, width=90):
    text = " ".join(str(text).split())
    return text if len(text) <= width else text[: width - 1] + "…"

def doc_text(doc_id):
    return documents[doc_id_to_index[doc_id]]["text"]

exercise = shortlists[0]
print("Query:", exercise["query"])
print(f"Gold:  {exercise['gold_doc_id']}  \"{preview(doc_text(exercise['gold_doc_id']))}\"")
print("Embedding shortlist (rank, document, cosine, candidate phrase):")
for row in exercise["candidates"]:
    marker = "  <-- GOLD" if row["is_gold"] else ""
    print(
        f"  {row['retrieval_rank']:>2}. {row['doc_id']:<12} score={row['retrieval_score']:.4f}  "
        f"\"{preview(doc_text(row['doc_id']))}\"{marker}"
    )

Query: I would like to transfer some money from my other bank account into this one.
Gold:  intent_065  "transfer into account"
Embedding shortlist (rank, document, cosine, candidate phrase):
   1. intent_065   score=0.7424  "transfer into account"  <-- GOLD
   2. intent_009   score=0.6794  "cancel transfer"
   3. intent_006   score=0.6489  "balance not updated after bank transfer"
   4. intent_056   score=0.6211  "top up by bank transfer charge"
   5. intent_049   score=0.6148  "pending transfer"
   6. intent_058   score=0.6059  "top up by cash or cheque"


## 12. Stage 2 — cross-encoder reranking

Qwen3-Reranker scores each query/document pair using the exact live-carrier contract:

- fixed system/user/assistant prompt;
- truncation to an 8,192-token prompt budget. The instruction, query and document form one text sequence, so tokens are removed from its end — the document comes last and loses tokens first; query and document are not truncated in balanced shares. The BYOD branch reports how many pairs were truncated;
- final-position logits for the tokens `no` and `yes`;
- two-way softmax;
- the `yes` share is the returned relevance score.

The score is **not a calibrated probability** and no threshold is applied. It is used only to order candidates supplied for the same query.

To reduce memory pressure, we keep the computed embedding vectors and remove the embedding model before loading the reranker.

**Infrastructure:** the collapsed class implements the pinned model contract. The learning question is whether joint query/document scoring changes the ordering. The returned score is relative evidence within a query’s shortlist, not a universal confidence value.

In [11]:
import gc
del embedder
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM

RERANK_MODEL_ID = RERANK_MANIFEST["modelId"]
RERANK_MODEL_REVISION = RERANK_MANIFEST["revision"]
RERANK_MAX_TOKENS = 8192
RERANK_MAX_PAIRS = 32
RERANK_BATCH_ATTENTION = RERANK_MAX_TOKENS ** 2  # same length-aware cap as the embedder (Section 8)
YES_TOKEN, NO_TOKEN = "yes", "no"
YES_TOKEN_ID, NO_TOKEN_ID = 9693, 2152

PREFIX = (
    "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the "
    'Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
)
SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

class RerankerRuntime:
    def __init__(self, root, device=DEVICE):
        self.device = device
        self.dtype = torch.bfloat16 if device.startswith("cuda") else torch.float32
        self.tokenizer = AutoTokenizer.from_pretrained(
            str(root),
            padding_side="left",
            trust_remote_code=False,
            local_files_only=True,
        )
        ids = (
            self.tokenizer.convert_tokens_to_ids(YES_TOKEN),
            self.tokenizer.convert_tokens_to_ids(NO_TOKEN),
        )
        if ids != (YES_TOKEN_ID, NO_TOKEN_ID):
            raise RuntimeError(f"Tokenizer maps yes/no to {ids}, expected {(YES_TOKEN_ID, NO_TOKEN_ID)}")

        self.model = AutoModelForCausalLM.from_pretrained(
            str(root),
            dtype=self.dtype,
            trust_remote_code=False,
            local_files_only=True,
        ).to(device).eval()

        self.prefix_ids = self.tokenizer.encode(PREFIX, add_special_tokens=False)
        self.suffix_ids = self.tokenizer.encode(SUFFIX, add_special_tokens=False)
        self.body_budget = RERANK_MAX_TOKENS - len(self.prefix_ids) - len(self.suffix_ids)

    @staticmethod
    def format_pair(query, document, instruction):
        return f"<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {document}"

    def score_batch(self, pairs, instruction):
        if not 1 <= len(pairs) <= RERANK_MAX_PAIRS:
            raise ValueError(f"Reranker batch must contain 1..{RERANK_MAX_PAIRS} pairs")
        bodies = []
        for i, (query, document) in enumerate(pairs):
            if not isinstance(query, str) or not query.strip():
                raise ValueError(f"pairs[{i}] query must be a non-empty string")
            if not isinstance(document, str) or not document.strip():
                raise ValueError(f"pairs[{i}] document must be a non-empty string")
            if len(query) > MAX_TEXT_CHARS or len(document) > MAX_TEXT_CHARS:
                raise ValueError(f"pairs[{i}] exceeds the character ceiling")
            bodies.append(self.format_pair(query, document, instruction))

        enc = self.tokenizer(
            bodies,
            padding=False,
            truncation="longest_first",
            return_attention_mask=False,
            max_length=self.body_budget,
        )
        enc["input_ids"] = [
            self.prefix_ids + row + self.suffix_ids
            for row in enc["input_ids"]
        ]
        batch = self.tokenizer.pad(enc, padding=True, return_tensors="pt").to(self.device)

        with torch.inference_mode():
            # Only the final position is scored: keep one logit row and no KV cache, so long prompts
            # do not materialize a (batch x length x vocabulary) logit tensor.
            last = self.model(**batch, logits_to_keep=1, use_cache=False).logits[:, -1, :]
        logits = torch.stack(
            [last[:, NO_TOKEN_ID], last[:, YES_TOKEN_ID]], dim=1
        ).float()
        probs = torch.softmax(logits, dim=1)[:, 1]
        counts = batch["attention_mask"].sum(dim=1).tolist()
        return probs.cpu().numpy().astype(np.float64), [int(x) for x in counts]

    def token_lengths(self, pairs, instruction):
        """Prompt token counts before truncation, for the truncation audit."""
        bodies = [self.format_pair(query, document, instruction) for query, document in pairs]
        encoded = self.tokenizer(bodies, padding=False, truncation=False, return_attention_mask=False)
        return [len(self.prefix_ids) + len(ids) + len(self.suffix_ids) for ids in encoded["input_ids"]]

    def score_all(self, pairs, instruction):
        lengths = [min(n, RERANK_MAX_TOKENS) for n in self.token_lengths(pairs, instruction)]
        scores, counts = [], []
        for start, end in attention_budget_batches(lengths, RERANK_MAX_PAIRS, RERANK_BATCH_ATTENTION):
            s, c = self.score_batch(pairs[start:end], instruction)
            scores.extend(float(x) for x in s)
            counts.extend(c)
        return scores, counts

reranker = RerankerRuntime(RERANK_DIR)
print({
    "model_id": RERANK_MODEL_ID,
    "revision": RERANK_MODEL_REVISION,
    "device": DEVICE,
    "score_kind": "relevance score, not a calibrated probability",
})

{'model_id': 'Qwen/Qwen3-Reranker-0.6B', 'revision': 'e61197ed45024b0ed8a2d74b80b4d909f1255473', 'device': 'cuda:0', 'score_kind': 'relevance score, not a calibrated probability'}


## 13. Rerank every top-*K* shortlist

For each query we score exactly the documents retrieved into its shortlist, preserve the original retrieval rank/score, then assign a new rank by descending reranker score.

No candidate is added after the first-stage retrieval boundary.

**Predict before running:** will the first shortlist’s gold document move up, down, or stay put? Record a reason from its text. **Expected result:** one score per candidate; the set of candidate IDs must stay unchanged even when their ranks change.

In [12]:
pairs = []
pair_meta = []

for shortlist in shortlists:
    for candidate in shortlist["candidates"]:
        doc = documents[doc_id_to_index[candidate["doc_id"]]]
        pairs.append((shortlist["query"], doc["text"]))
        pair_meta.append({
            "query_id": shortlist["query_id"],
            "doc_id": candidate["doc_id"],
            "retrieval_rank": candidate["retrieval_rank"],
            "retrieval_score": candidate["retrieval_score"],
            "is_gold": candidate["is_gold"],
        })

t0 = time.perf_counter()
reranker_scores, reranker_token_counts = reranker.score_all(pairs, RERANK_INSTRUCTION)
rerank_seconds = time.perf_counter() - t0

reranker_scores = check_relevance_scores(reranker_scores, len(queries) * RERANK_K, "Canonical reranking")

reranked_by_query = {}
reranked_rows = []
cursor = 0
for shortlist in shortlists:
    rows = []
    for _ in range(RERANK_K):
        meta = dict(pair_meta[cursor])
        meta["reranker_score"] = float(reranker_scores[cursor])
        rows.append(meta)
        cursor += 1
    rows.sort(key=lambda r: (-r["reranker_score"], r["retrieval_rank"]))
    for rank, row in enumerate(rows, start=1):
        row["reranked_rank"] = rank
        reranked_rows.append(dict(row))
    check_reranked_group(
        [c["doc_id"] for c in shortlist["candidates"]], rows, f"Reranked shortlist for {shortlist['query_id']}"
    )
    reranked_by_query[shortlist["query_id"]] = rows

print({
    "pairs_scored": len(pairs),
    "rerank_seconds": round(rerank_seconds, 3),
    "mean_seconds_per_pair": round(rerank_seconds / len(pairs), 6),
})

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'pairs_scored': 924, 'rerank_seconds': 37.194, 'mean_seconds_per_pair': 0.040253}


## 14. End-to-end evaluation

We report two different views.

### Full system

Every one of the 154 queries counts. If the gold document is absent from the shortlist:

- end-to-end recall receives no credit;
- reciprocal-rank contribution is `0`.

This prevents reranker evaluation from hiding first-stage retrieval failures.

### Conditional reranker

We also report ranking quality **only for queries where the gold document entered the shortlist**. These conditional metrics isolate what the second stage did with candidates it actually received, but they do not replace full-system metrics.

> **What to notice.** Compare the model result with the stated baseline/reference first. Then inspect the secondary metric or diagnostic that explains *how* the result was achieved; do not infer a universal model ranking from one sample and configuration.

**Checkpoint:** why might conditional MRR look strong while the full system performs poorly?

<details><summary>Sample answer</summary>
Conditional MRR excludes queries whose gold document never entered the shortlist. Full-system MRR includes those misses as zero. Report both denominators; otherwise a strong conditional result can conceal retrieval failures.
</details>

In [13]:
pipeline_gold_ranks = []
conditional_gold_ranks = []
effects = {"helped": 0, "hurt": 0, "unchanged": 0, "unrecoverable": 0}
effect_by_query = {}

for q, shortlist in zip(queries, shortlists, strict=True):
    qid = q["query_id"]
    before_gold = next((row["retrieval_rank"] for row in shortlist["candidates"] if row["is_gold"]), None)
    after_gold = next((row["reranked_rank"] for row in reranked_by_query[qid] if row["is_gold"]), None)

    if before_gold is None:
        effects["unrecoverable"] += 1
        effect_by_query[qid] = "unrecoverable"
        pipeline_gold_ranks.append(None)
        continue

    conditional_gold_ranks.append(after_gold)
    pipeline_gold_ranks.append(after_gold)

    if after_gold < before_gold:
        category = "helped"
    elif after_gold > before_gold:
        category = "hurt"
    else:
        category = "unchanged"
    effects[category] += 1
    effect_by_query[qid] = category

n_queries = len(queries)
composed = composed_metrics(pipeline_gold_ranks, RERANK_K, ks=(1, 3, 6))
pipeline_metrics = {
    "n_queries": n_queries,
    "rerank_k": RERANK_K,
    "shortlist_coverage@k": shortlist_coverage,
    **{key: value for key, value in composed.items() if key.startswith("recall@")},
    "mrr": composed["mrr"],
}

conditional_metrics = conditional_reranker_metrics(conditional_gold_ranks, RERANK_K)

if pipeline_metrics["recall@1"] > shortlist_coverage + 1e-12:
    raise RuntimeError("Pipeline recall@1 cannot exceed shortlist coverage")

effect_percent = {k: v / n_queries for k, v in effects.items()}

print("Pipeline metrics:", pipeline_metrics)
print("Conditional reranker metrics:", conditional_metrics)
print("Reranking effects:", effects)
print("Reranking effect percentages:", effect_percent)

Pipeline metrics: {'n_queries': 154, 'rerank_k': 6, 'shortlist_coverage@k': 0.9155844155844156, 'recall@1': 0.6233766233766234, 'recall@3': 0.8506493506493507, 'recall@6': 0.9155844155844156, 'mrr': 0.7378787878787879}
Conditional reranker metrics: {'n_queries': 141, 'n_documents': 6, 'mrr': 0.8059101654846336, 'median_rank': 1.0, 'recall@1': 0.6808510638297872, 'recall@3': 0.9290780141843972, 'recall@6': 1.0}
Reranking effects: {'helped': 29, 'hurt': 24, 'unchanged': 88, 'unrecoverable': 13}
Reranking effect percentages: {'helped': 0.18831168831168832, 'hurt': 0.15584415584415584, 'unchanged': 0.5714285714285714, 'unrecoverable': 0.08441558441558442}


## 15. Compare the systems

The rows below answer different questions:

- **Random:** what a uniform ordering would achieve in expectation.
- **Lexical:** what simple word overlap achieves without a neural model.
- **Qwen3 Embedding:** quality of the first-stage dense retriever over all 77 documents.
- **Embedding → Reranker:** quality of the composed system when the reranker sees only the top `RERANK_K`.

There is deliberately **no assertion that reranking must improve the result**. If this measured sample gets worse, that is a valid finding to inspect rather than something to tune away on the test set.

In [14]:
def fmt_pct(x):
    return f"{100*x:6.2f}%" if x is not None else "n/a"

# Fixed cutoffs are literal for every system. A selected K that is not already shown gets its own column.
TABLE_KS = [1, 3, 6] + ([RERANK_K] if RERANK_K not in (1, 3, 6) else [])

def table_row(system, pool, metrics):
    return {
        "system": system,
        "pool": pool,
        **{f"recall@{k}": metrics.get(f"recall@{k}") for k in TABLE_KS},
        "mrr": metrics["mrr"],
    }

summary_rows = [
    table_row("Random ranking (expected)", 77, random_metrics),
    table_row("Lexical Jaccard", 77, lexical_metrics),
    table_row("  diagnostic: ties against gold", 77, lexical_pessimistic_metrics),
    table_row("Qwen3 Embedding", 77, retriever_metrics),
    table_row(f"Embedding → Reranker top-{RERANK_K}", RERANK_K, pipeline_metrics),
]

header = "".join(f" {f'R@{k}':>9}" for k in TABLE_KS)
print(f"{'System':<34} {'Pool':>4}{header} {'MRR':>8}")
print("-" * (48 + 10 * len(TABLE_KS)))
for row in summary_rows:
    cells = "".join(f" {fmt_pct(row[f'recall@{k}']):>9}" for k in TABLE_KS)
    print(f"{row['system']:<34} {row['pool']:>4}{cells} {row['mrr']:>8.4f}")
print(
    f"\nLexical ties: the gold score is tied with another document for "
    f"{lexical_tie_diagnostic['queries_with_gold_score_tied']} of {len(queries)} queries. "
    "The main row scores the displayed ranking (ties keep document order); the diagnostic row ranks every tied document above gold."
)
if RERANK_K < 6:
    print(f"n/a: the composed system returns only its top-{RERANK_K} list, so deeper cutoffs are not measurable.")

System                             Pool       R@1       R@3       R@6      MRR
------------------------------------------------------------------------------
Random ranking (expected)            77     1.30%     3.90%     7.79%   0.0640
Lexical Jaccard                      77    35.06%    54.55%    68.83%   0.4846
  diagnostic: ties against gold      77    31.17%    48.70%    64.94%   0.4395
Qwen3 Embedding                      77    61.04%    81.17%    91.56%   0.7310
Embedding → Reranker top-6            6    62.34%    85.06%    91.56%   0.7379

Lexical ties: the gold score is tied with another document for 93 of 154 queries. The main row scores the displayed ranking (ties keep document order); the diagnostic row ranks every tied document above gold.


## 16. Failure analysis: where reranking helps and where it cannot

The notebook automatically selects one example from each available category:

- **helped** — gold rank improved;
- **hurt** — gold rank worsened;
- **unchanged** — gold rank stayed the same;
- **unrecoverable** — the retriever omitted the gold document from the shortlist.

This is often more informative than an aggregate metric because it exposes the boundary between candidate generation and candidate ordering.

In [15]:
query_by_id = {q["query_id"]: q for q in queries}
shortlist_by_id = {s["query_id"]: s for s in shortlists}
retrieval_by_id = {q["query_id"]: ranking for q, ranking in zip(queries, retrieval_rankings, strict=True)}

def show_case(query_id):
    q = query_by_id[query_id]
    category = effect_by_query[query_id]
    print("\n" + "=" * 90)
    print(f"CASE: {category.upper()}")
    print("Query:", q["query"])
    print(f"Gold:  {q['gold_doc_id']}  \"{preview(doc_text(q['gold_doc_id']))}\"")
    print("\nFirst-stage shortlist:")
    for row in shortlist_by_id[query_id]["candidates"]:
        marker = "  <-- GOLD" if row["is_gold"] else ""
        print(
            f"  {row['retrieval_rank']:>2}. {row['doc_id']:<12} cos={row['retrieval_score']:.4f}  "
            f"\"{preview(doc_text(row['doc_id']), 60)}\"{marker}"
        )
    print("\nReranked shortlist:")
    if category == "unrecoverable":
        gold_retrieval_rank = next(r["retrieval_rank"] for r in retrieval_by_id[query_id] if r["is_gold"])
        print(
            f"  Gold document was not supplied to the reranker "
            f"(first-stage rank {gold_retrieval_rank} of {len(documents)}, outside top-{RERANK_K})."
        )
    for row in reranked_by_query[query_id]:
        marker = "  <-- GOLD" if row["is_gold"] else ""
        print(
            f"  {row['reranked_rank']:>2}. {row['doc_id']:<12} rerank={row['reranker_score']:.4f} "
            f"(retrieval rank {row['retrieval_rank']})  \"{preview(doc_text(row['doc_id']), 60)}\"{marker}"
        )
    if category != "unrecoverable":
        before = next(r["retrieval_rank"] for r in shortlist_by_id[query_id]["candidates"] if r["is_gold"])
        after = next(r["reranked_rank"] for r in reranked_by_query[query_id] if r["is_gold"])
        print(f"\nGold rank: {before} → {after}")

for category in ("helped", "hurt", "unchanged", "unrecoverable"):
    match = next((qid for qid, value in effect_by_query.items() if value == category), None)
    if match is not None:
        show_case(match)
    else:
        print(f"\nNo {category!r} case occurred in this deterministic sample.")


CASE: HELPED
Query: I tried to top up using my card, but now the money just disappeared!
Gold:  intent_062  "topping up by card"

First-stage shortlist:
   1. intent_059   cos=0.7145  "top up failed"
   2. intent_061   cos=0.6935  "top up reverted"
   3. intent_062   cos=0.6875  "topping up by card"  <-- GOLD
   4. intent_048   cos=0.6840  "pending top up"
   5. intent_006   cos=0.6834  "balance not updated after bank transfer"
   6. intent_019   cos=0.6747  "card swallowed"

Reranked shortlist:
   1. intent_059   rerank=0.8872 (retrieval rank 1)  "top up failed"
   2. intent_062   rerank=0.7982 (retrieval rank 3)  "topping up by card"  <-- GOLD
   3. intent_019   rerank=0.7663 (retrieval rank 6)  "card swallowed"
   4. intent_048   rerank=0.2121 (retrieval rank 4)  "pending top up"
   5. intent_006   rerank=0.1192 (retrieval rank 5)  "balance not updated after bank transfer"
   6. intent_061   rerank=0.1067 (retrieval rank 2)  "top up reverted"

Gold rank: 3 → 2

CASE: HURT
Query: I 

## 17. Optional experiment — rerank depth versus compute

A larger shortlist gives the reranker access to more candidates and can increase the ceiling imposed by first-stage recall, but it also requires more cross-encoder pair evaluations.

The optional sweep evaluates `K = 3, 6, 10` by default. It is disabled on the canonical path because it multiplies reranker work and must not be used to tune `RERANK_K` on the same held-out queries reported above.

The worked Predict → Change → Run → Observe → Explain instructions are in “Try it yourself” below. The sweep does not assign new canonical metrics or change `RERANK_K`.

In [16]:
k_sweep_results = []

if RUN_K_SWEEP:
    for k in K_SWEEP_VALUES:
        sweep_pairs = []
        sweep_meta = []
        coverage = 0
        for q, ranking in zip(queries, retrieval_rankings, strict=True):
            topk = ranking[:k]
            coverage += any(row["is_gold"] for row in topk)
            for candidate in topk:
                doc = documents[doc_id_to_index[candidate["doc_id"]]]
                sweep_pairs.append((q["query"], doc["text"]))
                sweep_meta.append((q["query_id"], candidate["doc_id"], candidate["is_gold"]))

        t0 = time.perf_counter()
        sweep_scores, _ = reranker.score_all(sweep_pairs, RERANK_INSTRUCTION)
        wall = time.perf_counter() - t0
        sweep_scores = check_relevance_scores(sweep_scores, len(sweep_pairs), f"K-sweep K={k}")

        grouped = defaultdict(list)
        for meta, score in zip(sweep_meta, sweep_scores, strict=True):
            qid, doc_id, is_gold = meta
            grouped[qid].append((score, doc_id, is_gold, len(grouped[qid]) + 1))

        ranks = []
        for q in queries:
            # Same order as the canonical rerank: descending score, ties keep first-stage retrieval rank.
            rows = sorted(grouped[q["query_id"]], key=lambda x: (-x[0], x[3]))
            if len({row[1] for row in rows}) != len(rows):
                raise RuntimeError(f"K-sweep K={k}: duplicate candidates for {q['query_id']}")
            gold_rank = next((i + 1 for i, row in enumerate(rows) if row[2]), None)
            ranks.append(gold_rank)

        result = {
            "k": k,
            "shortlist_coverage": coverage / len(queries),
            "recall@1": sum(r == 1 for r in ranks) / len(ranks),
            "mrr": sum(0.0 if r is None else 1.0 / r for r in ranks) / len(ranks),
            "pairs": len(sweep_pairs),
            "reranker_seconds": wall,
        }
        k_sweep_results.append(result)

    print(f"{'K':>3} {'coverage':>10} {'R@1':>10} {'MRR':>9} {'pairs':>8} {'seconds':>10}")
    for row in k_sweep_results:
        print(
            f"{row['k']:>3} {fmt_pct(row['shortlist_coverage']):>10} "
            f"{fmt_pct(row['recall@1']):>10} {row['mrr']:>9.4f} "
            f"{row['pairs']:>8} {row['reranker_seconds']:>10.2f}"
        )
else:
    print("K sweep disabled on the canonical Run all path.")

K sweep disabled on the canonical Run all path.


## 18. Export machine-readable results and provenance

The notebook writes:

- `document_embeddings.npz`
- `retrieval_results.csv`
- `reranked_results.csv`
- `metrics.json`
- `provenance.json`
- `documents_lookup.csv` — document ID, intent phrase and intent label
- `queries_lookup.csv` — query ID, query text, gold document ID and gold phrase

Identifiers are preserved so downstream consumers can map every vector and ranking back to its source document/query.

Measured timings belong only to this execution environment.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [17]:
from datetime import datetime, timezone

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    out_dir / "document_embeddings.npz",
    doc_ids=np.array([d["doc_id"] for d in documents], dtype=str),
    embeddings=document_vectors.astype(np.float32),
)

with open(out_dir / "retrieval_results.csv", "w", encoding="utf-8", newline="") as handle:
    fieldnames = ["query_id", "doc_id", "retrieval_rank", "retrieval_score", "is_gold"]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(retrieval_rows)

with open(out_dir / "reranked_results.csv", "w", encoding="utf-8", newline="") as handle:
    fieldnames = [
        "query_id", "doc_id", "retrieval_rank", "retrieval_score",
        "reranked_rank", "reranker_score", "is_gold",
    ]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(reranked_rows)

# Lookups make every exported ID readable offline: document phrases, query text and gold labels.
with open(out_dir / "documents_lookup.csv", "w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["doc_id", "text", "intent"])
    writer.writeheader()
    writer.writerows({"doc_id": d["doc_id"], "text": d["text"], "intent": d["intent"]} for d in documents)

with open(out_dir / "queries_lookup.csv", "w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["query_id", "query", "gold_doc_id", "gold_text", "intent"])
    writer.writeheader()
    writer.writerows({
        "query_id": q["query_id"],
        "query": q["query"],
        "gold_doc_id": q["gold_doc_id"],
        "gold_text": doc_text(q["gold_doc_id"]),
        "intent": q["intent"],
    } for q in queries)

metrics_export = {
    "random": random_metrics,
    "lexical": lexical_metrics,
    "lexical_ties_against_gold_diagnostic": lexical_pessimistic_metrics,
    "lexical_tie_diagnostic": lexical_tie_diagnostic,
    "retriever": retriever_metrics,
    "shortlist": {
        "k": RERANK_K,
        "coverage": shortlist_coverage,
    },
    "pipeline": pipeline_metrics,
    "conditional_reranker": conditional_metrics,
    "effects": {
        "counts": effects,
        "fractions": effect_percent,
    },
    "timings_seconds": {
        "document_index": index_seconds,
        "query_embedding": query_embed_seconds,
        "retrieval_matrix": retrieval_seconds,
        "reranking": rerank_seconds,
    },
    "k_sweep": k_sweep_results,
}
(out_dir / "metrics.json").write_text(
    json.dumps(metrics_export, indent=2),
    encoding="utf-8",
)

NOTEBOOK_IDENTITY = {
    "repository": "kurtvalcorza/qwen3-embedding-pipeline",
    "path": "tutorials/DIMER_Qwen3_Semantic_Search_Reranking_Workshop.ipynb",
    "source": "tools/semantic_search_reranking_workshop_source.py",
    "generator": "tools/build_semantic_search_reranking_workshop.py",
}

provenance = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": NOTEBOOK_IDENTITY,
    "notebook_spec": "2.1",
    "profile": "MULTI-CAPABILITY",
    "pedagogical_mode": "WORKSHOP",
    "standalone": True,
    "embedding_model": {
        "id": EMBED_MODEL_ID,
        "revision": EMBED_MODEL_REVISION,
        "model_safetensors_sha256": next(
            x["sha256"] for x in EMBED_MANIFEST["files"] if x["path"] == "model.safetensors"
        ),
        "instruction": EMBEDDING_INSTRUCTION,
    },
    "reranker_model": {
        "id": RERANK_MODEL_ID,
        "revision": RERANK_MODEL_REVISION,
        "model_safetensors_sha256": next(
            x["sha256"] for x in RERANK_MANIFEST["files"] if x["path"] == "model.safetensors"
        ),
        "instruction": RERANK_INSTRUCTION,
        "score_kind": "relevance score, not a calibrated probability",
    },
    "dataset": {
        "name": CORPUS_NAME,
        "license": CORPUS_LICENSE,
        "release": CORPUS_RELEASE,
        "source_files": {
            split: {"file": spec[0], "bytes": spec[1], "sha256": spec[2]}
            for split, spec in CORPUS_FILES.items()
        },
        "sample_seed": SAMPLE_SEED,
        "sample_digest": problem_manifest["digest"],
        "documents": len(documents),
        "queries": len(queries),
    },
    "configuration": {
        "rerank_k": RERANK_K,
        "run_k_sweep": RUN_K_SWEEP,
        "tie_policy": "displayed ranking: descending score, exact ties keep document order (rerank ties keep retrieval rank)",
    },
    "max_token_counts": {
        "documents": max(document_token_counts),
        "queries": max(query_token_counts),
        "reranker_prompts": max(reranker_token_counts),
        "budget": EMBED_MAX_TOKENS,
    },
    "runtime": RUNTIME,
    "timings_seconds": metrics_export["timings_seconds"],
}
(out_dir / "provenance.json").write_text(
    json.dumps(provenance, indent=2),
    encoding="utf-8",
)

print("Wrote:")
for path in sorted(out_dir.iterdir()):
    print(" ", path)

Wrote:
  outputs/qwen3_semantic_search/document_embeddings.npz
  outputs/qwen3_semantic_search/documents_lookup.csv
  outputs/qwen3_semantic_search/metrics.json
  outputs/qwen3_semantic_search/provenance.json
  outputs/qwen3_semantic_search/queries_lookup.csv
  outputs/qwen3_semantic_search/reranked_results.csv
  outputs/qwen3_semantic_search/retrieval_results.csv


## 19. Bring Your Own Data (optional)

The canonical path above is complete. BYOD is optional and disabled by default.

### Documents CSV

Required columns:

`doc_id,text`

Constraints:

- 2–1,000 documents;
- unique non-empty `doc_id`;
- non-empty text;
- each document ≤ 100,000 characters.

The notebook does **not** chunk long documents automatically. Pre-chunk them into meaningful retrievable units.

The files are read strictly as UTF-8 CSV (a spreadsheet byte-order mark is accepted): header names must be unique and non-empty, and every row must have exactly as many fields as the header — quote any value that contains a comma. Errors name the file, line and column, and are raised before any model is loaded.

### Queries CSV

Required columns:

`query_id,query`

Optional evaluation column:

`gold_doc_id`

If all queries contain a valid `gold_doc_id`, the branch reports retrieval/reranking metrics. Without labels, the evaluation verdict is `not-measurable`, but rankings are still produced.

### Privacy

This standalone workflow processes supplied text inside the selected notebook runtime and does not intentionally transmit it to DIMER services. Do not upload confidential, restricted, sensitive, personal, regulated, or otherwise unauthorized material to a hosted notebook environment.

### Evaluation and exports

Supply a valid `gold_doc_id` for every query, or leave it blank for every query. Mixed labelled/unlabelled input is refused so labels are not silently discarded. Every supplied gold ID must name a document. Query count is limited to 1–500; queries have the same 100,000-character limit as documents.

### Instructions

Both models are instruction-conditioned. The canonical instructions describe the Banking77 task (a customer message matched to the banking intent it expresses). For any other task, set `BYOD_EMBEDDING_INSTRUCTION` and `BYOD_RERANK_INSTRUCTION` in Configuration to one sentence describing your query → document relationship, for example *“Given a question, retrieve the policy passage that answers it.”* Left blank, they reuse the banking-intent instructions and the branch prints a note; `byod_results.json` records the instructions used and whether they were user-supplied.

### Truncation

Both models apply an 8,192-token budget, including their instruction/prompt overhead. Text below the character ceiling may still be truncated; pre-chunk it when preserving the full document matters. `byod_results.json` reports, for documents, queries and reranker pairs, how many inputs were truncated and their token counts before and after truncation. Batches are sized by attention cost (inputs × longest length², capped at one 8,192-token input), so long inputs are processed a few at a time — a full-length document or pair runs alone — instead of in full 32- or 64-item batches; expect long documents to take noticeably longer.

The embeddings, similarity rankings and reranker scores go through the same shape, finiteness, norm, count, range and permutation checks as the canonical path before anything is exported.

The branch unloads the canonical reranker before loading the BYOD embedder, then unloads that embedder before reloading the reranker. It writes `byod_results.json` and `byod_rankings.csv` separately from the canonical exports, with input-file digests, effective K, model revisions and runtime. These files contain supplied text identifiers and rankings; keep them with the corresponding inputs.

In [18]:
byod_result = None

def load_csv_records(path, required):
    """Read a UTF-8 CSV strictly: unique header names, every row as wide as the header.

    Returns (rows, line_numbers) so later checks can name the file, line and column of a bad value.
    """
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"File not found: {p}")
    with open(p, encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader, None)
        if header is None:
            raise ValueError(f"{p} is empty")
        names = [name.strip() for name in header]
        if any(not name for name in names):
            raise ValueError(f"{p}, line 1: column {names.index('') + 1} has an empty header name")
        duplicates = sorted({name for name in names if names.count(name) > 1})
        if duplicates:
            raise ValueError(f"{p}, line 1: duplicate column names {duplicates}")
        missing = set(required) - set(names)
        if missing:
            raise ValueError(f"{p} is missing columns {sorted(missing)}")
        rows, lines = [], []
        for record in reader:
            if not record:
                continue  # blank line
            if len(record) != len(names):
                raise ValueError(
                    f"{p}, line {reader.line_num}: expected {len(names)} fields ({', '.join(names)}), "
                    f"found {len(record)}; quote values that contain commas"
                )
            rows.append(dict(zip(names, record)))
            lines.append(reader.line_num)
    if not rows:
        raise ValueError(f"{p} has a header but no data rows")
    return rows, lines

def repeated(values):
    seen, dups = set(), []
    for value in values:
        if value in seen and value not in dups:
            dups.append(value)
        seen.add(value)
    return dups

def truncation_summary(ids, tokens_before, tokens_kept, budget):
    cut = [(i, b, k) for i, b, k in zip(ids, tokens_before, tokens_kept, strict=True) if b > k]
    return {
        "inputs": len(ids),
        "truncated": len(cut),
        "budget_tokens": budget,
        "examples": [{"id": i, "tokens_before": b, "tokens_kept": k} for i, b, k in cut[:20]],
    }

if USE_BYOD:
    if not DOCUMENTS_PATH or not QUERIES_PATH:
        raise ValueError("Set DOCUMENTS_PATH and QUERIES_PATH when USE_BYOD=True")

    byod_docs, doc_lines = load_csv_records(DOCUMENTS_PATH, {"doc_id", "text"})
    byod_queries, query_lines = load_csv_records(QUERIES_PATH, {"query_id", "query"})

    if not 2 <= len(byod_docs) <= 1000:
        raise ValueError(f"{DOCUMENTS_PATH}: BYOD documents must contain 2..1000 rows, found {len(byod_docs)}")
    if not 1 <= len(byod_queries) <= 500:
        raise ValueError(f"{QUERIES_PATH}: BYOD queries must contain 1..500 rows, found {len(byod_queries)}")

    doc_ids = [r["doc_id"].strip() for r in byod_docs]
    query_ids = [r["query_id"].strip() for r in byod_queries]
    for path, lines, ids, column in (
        (DOCUMENTS_PATH, doc_lines, doc_ids, "doc_id"),
        (QUERIES_PATH, query_lines, query_ids, "query_id"),
    ):
        blank = [line for line, value in zip(lines, ids) if not value]
        if blank:
            raise ValueError(f"{path}, line {blank[0]}, column {column!r}: BYOD {column} values must be unique and non-empty")
        dups = repeated(ids)
        if dups:
            raise ValueError(f"{path}, column {column!r}: BYOD {column} values must be unique and non-empty; repeated {dups[:10]}")

    for path, rows, lines, column in (
        (DOCUMENTS_PATH, byod_docs, doc_lines, "text"),
        (QUERIES_PATH, byod_queries, query_lines, "query"),
    ):
        for row, line in zip(rows, lines):
            if not row[column].strip() or len(row[column]) > MAX_TEXT_CHARS:
                raise ValueError(
                    f"{path}, line {line}, column {column!r}: empty or longer than {MAX_TEXT_CHARS} characters"
                )

    gold_ids = [(row.get("gold_doc_id") or "").strip() for row in byod_queries]
    if any(gold_ids) and not all(gold_ids):
        first_blank = next(line for line, value in zip(query_lines, gold_ids) if not value)
        raise ValueError(
            f"{QUERIES_PATH}, line {first_blank}: supply gold_doc_id for every query or leave every gold_doc_id blank"
        )
    labelled = all(gold_ids)
    if labelled:
        known = set(doc_ids)
        unknown = sorted({g for g in gold_ids if g not in known})
        if unknown:
            raise ValueError(f"{QUERIES_PATH}: BYOD queries reference unknown gold_doc_id values: {unknown[:10]}")

    # Instructions describe the retrieval task. Blank BYOD values reuse the banking-intent instructions,
    # which are appropriate only when your data poses the same task.
    byod_embedding_instruction = BYOD_EMBEDDING_INSTRUCTION.strip() or EMBEDDING_INSTRUCTION
    byod_rerank_instruction = BYOD_RERANK_INSTRUCTION.strip() or RERANK_INSTRUCTION
    byod_instruction_source = {
        "embedding": "user-supplied" if BYOD_EMBEDDING_INSTRUCTION.strip() else "default banking-intent task",
        "reranker": "user-supplied" if BYOD_RERANK_INSTRUCTION.strip() else "default banking-intent task",
    }
    if "default banking-intent task" in byod_instruction_source.values():
        print(
            "Note: BYOD is using the banking-intent instruction(s) "
            f"{sorted(k for k, v in byod_instruction_source.items() if v != 'user-supplied')}. "
            "Set BYOD_EMBEDDING_INSTRUCTION / BYOD_RERANK_INSTRUCTION to describe a different retrieval task."
        )

    byod_doc_texts = [r["text"].strip() for r in byod_docs]
    byod_query_texts = [r["query"].strip() for r in byod_queries]

    # Keep only one model resident at a time, including this optional branch.
    globals().pop("reranker", None)
    globals().pop("byod_embedder", None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # Reload the embedder because the canonical path intentionally released it before reranking.
    try:
        byod_embedder = EmbeddingRuntime(EMBED_DIR)
        byod_doc_vectors, byod_doc_kept = byod_embedder.embed_all(
            byod_doc_texts, "document", byod_embedding_instruction
        )
        byod_query_vectors, byod_query_kept = byod_embedder.embed_all(
            byod_query_texts, "query", byod_embedding_instruction
        )
        byod_doc_before = byod_embedder.token_lengths(byod_doc_texts, "document", byod_embedding_instruction)
        byod_query_before = byod_embedder.token_lengths(byod_query_texts, "query", byod_embedding_instruction)
        check_embedding_matrix(byod_doc_vectors, len(byod_docs), EMBED_DIM, "BYOD document embeddings")
        check_embedding_matrix(byod_query_vectors, len(byod_queries), EMBED_DIM, "BYOD query embeddings")
        byod_sim = byod_query_vectors @ byod_doc_vectors.T
    except Exception as exc:
        # Failed model frames can retain tensors/self even after deleting the global name.
        import traceback
        traceback.clear_frames(exc.__traceback__)
        raise
    finally:
        globals().pop("byod_embedder", None)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        # Restore the reranker even if optional embedding failed, so retry/sweep remains usable.
        reranker = RerankerRuntime(RERANK_DIR)

    effective_k = min(RERANK_K, len(byod_docs))
    byod_shortlists = []
    byod_retrieval_ranks = []
    for qi, row in enumerate(byod_queries):
        order = check_permutation(
            np.argsort(-byod_sim[qi], kind="stable"), len(byod_docs), f"BYOD retrieval for {query_ids[qi]}"
        )
        if labelled:
            byod_retrieval_ranks.append(order.index(doc_ids.index(gold_ids[qi])) + 1)
        byod_shortlists.append(order[:effective_k])

    byod_pairs = []
    byod_pair_ids = []
    for qi, row in enumerate(byod_queries):
        for idx in byod_shortlists[qi]:
            byod_pairs.append((byod_query_texts[qi], byod_doc_texts[idx]))
            byod_pair_ids.append(f"{query_ids[qi]}|{doc_ids[idx]}")

    byod_scores, byod_pair_kept = reranker.score_all(byod_pairs, byod_rerank_instruction)
    byod_scores = check_relevance_scores(byod_scores, len(byod_pairs), "BYOD reranking")
    byod_pair_before = reranker.token_lengths(byod_pairs, byod_rerank_instruction)

    byod_rows = []
    cursor = 0
    byod_final_ranks = []
    for qi, row in enumerate(byod_queries):
        items = []
        for retrieval_rank, idx in enumerate(byod_shortlists[qi], start=1):
            items.append({
                "query_id": query_ids[qi],
                "doc_id": doc_ids[idx],
                "retrieval_rank": retrieval_rank,
                "retrieval_score": float(byod_sim[qi, idx]),
                "reranker_score": byod_scores[cursor],
                "is_gold": labelled and doc_ids[idx] == gold_ids[qi],
            })
            cursor += 1
        items.sort(key=lambda x: (-x["reranker_score"], x["retrieval_rank"]))
        for reranked_rank, item in enumerate(items, start=1):
            item["reranked_rank"] = reranked_rank
            byod_rows.append(item)
        check_reranked_group([doc_ids[i] for i in byod_shortlists[qi]], items, f"BYOD rerank for {query_ids[qi]}")
        if labelled:
            byod_final_ranks.append(
                next((x["reranked_rank"] for x in items if x["is_gold"]), None)
            )

    byod_result = {
        "documents": len(byod_docs),
        "queries": len(byod_queries),
        "labelled": labelled,
        "evaluation_verdict": "measured" if labelled else "not-measurable",
        "rows": byod_rows,
    }

    if labelled:
        byod_result["retriever_metrics"] = metrics_from_ranks(
            byod_retrieval_ranks, len(byod_docs), ks=tuple(sorted({1, 3, effective_k}))
        )
        byod_result["pipeline_metrics"] = composed_metrics(byod_final_ranks, effective_k, ks=(1, 3))
        byod_result["pipeline_recall@1"] = byod_result["pipeline_metrics"]["recall@1"]
        byod_result["pipeline_mrr"] = byod_result["pipeline_metrics"]["mrr"]
    else:
        byod_result["reason"] = "gold_doc_id was not supplied for every query"

    byod_result["truncation"] = {
        "documents": truncation_summary(doc_ids, byod_doc_before, byod_doc_kept, EMBED_MAX_TOKENS),
        "queries": truncation_summary(query_ids, byod_query_before, byod_query_kept, EMBED_MAX_TOKENS),
        "reranker_pairs": truncation_summary(byod_pair_ids, byod_pair_before, byod_pair_kept, RERANK_MAX_TOKENS),
    }

    byod_result["provenance"] = {
        "documents_sha256": hashlib.sha256(Path(DOCUMENTS_PATH).read_bytes()).hexdigest(),
        "queries_sha256": hashlib.sha256(Path(QUERIES_PATH).read_bytes()).hexdigest(),
        "effective_rerank_k": effective_k,
        "embedding_revision": EMBED_MODEL_REVISION,
        "reranker_revision": RERANK_MODEL_REVISION,
        "embedding_instruction": byod_embedding_instruction,
        "rerank_instruction": byod_rerank_instruction,
        "instruction_source": byod_instruction_source,
        "runtime": RUNTIME,
    }
    byod_out_dir = Path(OUTPUT_DIR)
    byod_out_dir.mkdir(parents=True, exist_ok=True)
    (byod_out_dir / "byod_results.json").write_text(
        json.dumps(byod_result, indent=2), encoding="utf-8"
    )
    with open(byod_out_dir / "byod_rankings.csv", "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(byod_rows[0]))
        writer.writeheader()
        writer.writerows(byod_rows)

    print({
        key: value for key, value in byod_result.items()
        if key not in {"rows", "truncation"}
    })
    print("Truncated inputs:", {name: part["truncated"] for name, part in byod_result["truncation"].items()})
else:
    print("BYOD disabled on the canonical Run all path.")

BYOD disabled on the canonical Run all path.


## 20. Interpretation, limitations, and transfer

### Retrieval and reranking solve different problems

Dense retrieval efficiently searches a corpus using reusable document representations. Cross-encoder reranking spends more computation on query–candidate interactions after the search space has already been reduced.

### Reranking does not repair missing candidates

First-stage `recall@K` / shortlist coverage is a hard ceiling for a reranker restricted to that shortlist.

### Scores are not calibrated probabilities

Embedding cosine similarity and Qwen3 reranker relevance scores are ranking signals. Their numeric values should not be interpreted as universal confidence or probability.

### Banking77 is tutorial evidence

This notebook has:

- 77 short intent phrases;
- one gold intent per query;
- 154 deterministic held-out queries.

It does **not** establish performance on long documents, multilingual enterprise corpora, graded relevance, large-scale indexing, or production traffic.

### Pretraining overlap is unknown

This notebook cannot establish that Banking77 or equivalent text was absent from upstream pretraining.

### Real retrieval can have multiple relevant documents

A real information-retrieval evaluation commonly needs binary or graded relevance judgements for several candidates per query. The one-label Banking77 construction is simpler.

### Scaling

Document embeddings can be precomputed. Cross-encoder work scales approximately with:

`number of queries × rerank depth`

That cost structure is the reason rerankers usually sit behind a first-stage retriever.

### Next experiment

A clean follow-on is a separate grounded-generation notebook that consumes these ranked results and adds an instruction-following language model. Keeping generation separate here makes retrieval quality measurable on its own.

## 21. Terminal summary

The final cell checks that all required exports exist and prints the measured results from this execution.

In [19]:
required_outputs = [
    out_dir / "document_embeddings.npz",
    out_dir / "retrieval_results.csv",
    out_dir / "reranked_results.csv",
    out_dir / "metrics.json",
    out_dir / "provenance.json",
    out_dir / "documents_lookup.csv",
    out_dir / "queries_lookup.csv",
]
if USE_BYOD:
    required_outputs.extend([out_dir / "byod_results.json", out_dir / "byod_rankings.csv"])
missing = [str(path) for path in required_outputs if not path.is_file()]
if missing:
    raise RuntimeError(f"Required outputs missing: {missing}")

print("DIMER Qwen3 Semantic Search + Reranking Notebook")
print("-" * 55)
print(f"Queries evaluated: {len(queries)}")
print(f"Documents indexed: {len(documents)}")
print(f"Rerank depth: {RERANK_K}")
print()
print(f"Lexical recall@1:              {lexical_metrics['recall@1']:.4f}")
print(f"  (ties against gold, diag.):  {lexical_pessimistic_metrics['recall@1']:.4f}")
print(f"Embedding recall@1:            {retriever_metrics['recall@1']:.4f}")
print(f"Embedding recall@{RERANK_K}:            {retriever_metrics[f'recall@{RERANK_K}']:.4f}")
print(f"Shortlist coverage@{RERANK_K}:          {shortlist_coverage:.4f}")
print(f"Two-stage recall@1:            {pipeline_metrics['recall@1']:.4f}")
print(f"Two-stage MRR:                 {pipeline_metrics['mrr']:.4f}")
print()
print("Reranking effects:")
for key in ("helped", "hurt", "unchanged", "unrecoverable"):
    print(f"  {key:<13}: {effects[key]:>3} ({effect_percent[key]:.1%})")
print()
print(f"Outputs: {out_dir}/")
print("Required output files:", len(required_outputs))
if byod_result is not None:
    # Your own data is a different dataset: report it separately from the canonical Banking77 results.
    print()
    print("BYOD (your data — not comparable with the Banking77 numbers above)")
    print("-" * 55)
    print(f"Documents: {byod_result['documents']}   Queries: {byod_result['queries']}   "
          f"Effective K: {byod_result['provenance']['effective_rerank_k']}")
    print(f"Evaluation: {byod_result['evaluation_verdict']}")
    if byod_result["labelled"]:
        print(f"Embedding recall@1:            {byod_result['retriever_metrics']['recall@1']:.4f}")
        print(f"Two-stage recall@1:            {byod_result['pipeline_recall@1']:.4f}")
        print(f"Two-stage MRR:                 {byod_result['pipeline_mrr']:.4f}")
    print("Instructions:", byod_result["provenance"]["instruction_source"])
    print("Truncated inputs:", {k: v["truncated"] for k, v in byod_result["truncation"].items()})
    print("Files: byod_results.json, byod_rankings.csv")

DIMER Qwen3 Semantic Search + Reranking Notebook
-------------------------------------------------------
Queries evaluated: 154
Documents indexed: 77
Rerank depth: 6

Lexical recall@1:              0.3506
  (ties against gold, diag.):  0.3117
Embedding recall@1:            0.6104
Embedding recall@6:            0.9156
Shortlist coverage@6:          0.9156
Two-stage recall@1:            0.6234
Two-stage MRR:                 0.7379

Reranking effects:
  helped       :  29 (18.8%)
  hurt         :  24 (15.6%)
  unchanged    :  88 (57.1%)
  unrecoverable:  13 (8.4%)

Outputs: outputs/qwen3_semantic_search/
Required output files: 7


## Try it yourself — one controlled change

Use Section 17’s optional depth sweep; do not restart the canonical pipeline or change its `RERANK_K`.

1. **Predict:** write whether depths 3, 6 and 10 will increase shortlist coverage, pair count and end-to-end recall@1. Coverage cannot decrease with a longer prefix, but ranking quality need not improve.
2. **Change one thing:** after the canonical run, set `RUN_K_SWEEP = True` in a new scratch cell (do not rerun Configuration). Keep `K_SWEEP_VALUES=[3, 6, 10]`, the model, instructions, queries and first-stage rankings unchanged.
3. **Run:** execute only the Section 17 sweep cell. It writes a separate `k_sweep_results` list, leaving the canonical rankings and metrics intact.
4. **Observe:** record K, coverage, recall@1, MRR, number of pairs and measured seconds. Compare each prediction with the result; timings also include runtime variability.
5. **Explain:** “I predicted __. At K=__ versus __, coverage changed from __ to __, recall@1 from __ to __, and pairs from __ to __. This suggests __, but one limitation is __.”

To retain the sweep, rerun only Section 18 export afterward; it retains canonical metrics and adds the exploratory sweep. Keep the original report if you need the pre-exercise record. These are the same previously inspected test queries: do not select a new canonical K from them and call it an unbiased improvement. Confirm any selected setting on a separate untouched query set.

## Self-paced checkpoint

Before reading the sample interpretation, answer in your own words:

1. What was the model or system asked to do?
2. Which baseline/reference tells you whether the model added useful capability?
3. What is the most important failure mode or tradeoff visible in this notebook?
4. What would you need to test before applying the result to a different dataset or operational setting?

<details>
<summary><b>Show a sample interpretation</b></summary>

A reranker can improve ordering only inside the shortlist it receives. If the relevant document is absent after first-stage retrieval, reranking cannot recover it. Compare first-stage recall with end-to-end ranking metrics before attributing a failure to the reranker.

A complete answer should cite the outputs from **your run**, because small numerical differences can occur across supported runtimes.

</details>


## Write an evidence-based conclusion

Use the outputs from your run rather than declaring a universal winner.

1. **State the question.** What capability or comparison did this notebook test?
2. **Report the primary result.** Compare the relevant model/system with its baseline or reference on the held-out or otherwise designated evaluation data.
3. **Add supporting evidence.** Include the most informative secondary metric, error pattern, qualitative diagnostic, or disagreement.
4. **Account for cost or complexity.** Mention runtime, model footprint, extra stages, or adaptation when they materially affect the comparison.
5. **State the limits.** Say what this dataset, split, model revision, and configuration do—and do not—support.

State whether two-stage retrieval improved the held-out ranking relative to the non-neural and embedding-only references. Report the principal retrieval metric(s), note whether the relevant item was present in the first-stage shortlist, describe the compute tradeoff introduced by reranking, and limit the conclusion to this Banking77-derived tutorial task.


# Troubleshooting

| What you see | Likely cause | What to do |
|---|---|---|
| Accelerator unavailable or the model is unexpectedly slow | The runtime does not match the documented resource envelope | Select the documented accelerator/runtime, start a fresh session, and run the notebook top-to-bottom. |
| Setup stops with a Linux x86_64 message, or a uv-wheel or lock digest error | The runtime is not Linux x86_64, or a download was incomplete or altered | Use Google Colab, Kaggle or Linux Jupyter on x86_64 and run the setup cell again. Do not edit the pins or the carried lock; regenerate the notebook from the repository instead. |
| "The isolated environment's Python process exited" | The worker process running the cells stopped, usually out of memory | Restart the session and choose **Run all** again with the defaults. |
| Model/sample digest or size verification fails | A download is incomplete or upstream bytes differ from the pinned artifact | Remove the affected runtime cache/download and rerun. Do not disable the integrity check. |
| Out-of-memory / session restart | Too many large models or intermediate objects are resident | Keep the defaults (`RERANK_K = 6`, sweep and BYOD off), follow the notebook's unload steps, and run the optional K-sweep and BYOD branches one at a time. For BYOD, pre-chunk long documents. |
| BYOD validation fails | User input does not satisfy the documented schema, shape, labels, or limits | Read the validation message, correct the stated field/shape/format, and rerun the BYOD branch. |
| Your numbers differ slightly from the example/expectation | Supported hardware or library execution can introduce small numerical variation | Compare the qualitative pattern, metric definitions, split, and model revision before treating the difference as meaningful. |


# Glossary

| Term | Meaning in this notebook |
|---|---|
| **Embedding** | A numerical vector representation used here to compare semantic similarity. |
| **Bi-encoder / retriever** | Encodes queries and documents separately so document vectors can be reused. |
| **Cosine similarity** | A similarity measure based on the angle between vectors. |
| **Top-k** | The first k candidates returned by the retriever. |
| **Cross-encoder / reranker** | Scores a query and candidate jointly, using more compute for a smaller shortlist. |
| **Recall@k** | Whether the relevant item appears within the first k retrieved candidates. |

**MRR:** mean reciprocal rank; rank 1 contributes 1, rank 4 contributes 0.25, and a missing gold document contributes zero to the full system. **Shortlist coverage:** fraction of queries whose gold document enters top-K. **Conditional metric:** calculated only for that covered subset. **Jaccard:** shared word types divided by all distinct word types across query and document.